In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
# Cell 1 — Install all dependencies

!pip install -q \
    chromadb \
    sentence-transformers \
    pymupdf \
    fastapi \
    uvicorn \
    pydantic \
    pytest \
    nest-asyncio

print("All packages installed")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 39.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently

In [3]:
# Cell 2 — Create directory structure

import os

BASE = "/kaggle/working/neva"

dirs = [
    f"{BASE}/rag",
    f"{BASE}/data/raw",
    f"{BASE}/data/chunks",
    f"{BASE}/chroma_db",
    f"{BASE}/tests",
    f"{BASE}/docs",
]

for d in dirs:
    os.makedirs(d, exist_ok=True)

# Make rag/ a package
open(f"{BASE}/rag/__init__.py", "w").close()
open(f"{BASE}/tests/__init__.py", "w").close()

print("✓ Project structure created")
print("\nDirectory tree:")
for root, dirs_list, files in os.walk(BASE):
    level = root.replace(BASE, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    for f in files:
        print(f"{indent}  {f}")

✓ Project structure created

Directory tree:
neva/
  rag/
    __init__.py
  docs/
  data/
    chunks/
    raw/
  chroma_db/
  tests/
    __init__.py


In [4]:
# Cell 3 — Write rag/models.py

content = '''"""
NEVA RAG — Pydantic models.
Contract between: chunker → ChromaDB → retriever → Gemma prompt builder.
"""

from __future__ import annotations
from typing import Literal, Optional
from pydantic import BaseModel, Field


# ── Type aliases ──────────────────────────────────────────────────────────────

StepType     = Literal["overview", "assessment", "action", "warning", "do_not"]
AgeGroup     = Literal["adult", "paediatric", "both"]
LanguageCode = Literal["en", "ne"]
Source       = Literal[
    "WHO_BEC_2016",
    "Nepal_MoHP_2078",
    "WHO_PHEC_2026",
    "NEVA_SEED",
]


# ── Chunk metadata (stored flat in ChromaDB) ──────────────────────────────────

class ChunkMetadata(BaseModel):
    chunk_id:       str
    condition:      str             = Field(..., description="Primary condition slug e.g. 'choking'")
    condition_tags: list[str]       = Field(default_factory=list)
    urgency_level:  int             = Field(..., ge=1, le=5)
    age_group:      AgeGroup
    language:       LanguageCode
    source:         Source
    step_type:      StepType
    section:        str
    page_ref:       Optional[str]   = None
    keywords:       list[str]       = Field(default_factory=list)

    def to_chroma_meta(self) -> dict:
        """ChromaDB only accepts flat str / int / float / bool."""
        d = self.model_dump()
        d["condition_tags"] = "|".join(d["condition_tags"])
        d["keywords"]       = "|".join(d["keywords"])
        return d


# ── Protocol chunk (unit of storage) ─────────────────────────────────────────

class ProtocolChunk(BaseModel):
    metadata: ChunkMetadata
    text:     str


# ── Retrieval request (from extraction stage) ─────────────────────────────────

class RetrievalRequest(BaseModel):
    query:         str
    condition:     Optional[str]      = None
    urgency_level: Optional[int]      = Field(None, ge=1, le=5)
    age_group:     Optional[AgeGroup] = None
    language:      LanguageCode       = "en"
    top_k:         int                = Field(5, ge=1, le=15)


# ── Retrieval result (injected into Gemma prompt) ─────────────────────────────

class RetrievedChunk(BaseModel):
    chunk_id:      str
    condition:     str
    urgency_level: int
    step_type:     StepType
    source:        str
    text:          str
    score:         float   # cosine distance — lower = more similar

class RetrievalResult(BaseModel):
    query:    str
    chunks:   list[RetrievedChunk]
    grounded: bool   # True if at least one chunk scored below threshold
'''

with open("/kaggle/working/neva/rag/models.py", "w") as f:
    f.write(content)

# Verify import works
import sys
sys.path.insert(0, "/kaggle/working/neva")

from rag.models import (
    ChunkMetadata, ProtocolChunk,
    RetrievalRequest, RetrievalResult, RetrievedChunk
)

print("✓ models.py written and imported successfully")

# Quick schema check
test_meta = ChunkMetadata(
    chunk_id       = "test_001",
    condition      = "choking",
    condition_tags = ["airway", "obstruction"],
    urgency_level  = 5,
    age_group      = "adult",
    language       = "en",
    source         = "WHO_BEC_2016",
    step_type      = "action",
    section        = "Choking Treatment",
    keywords       = ["choking", "heimlich"],
)
flat = test_meta.to_chroma_meta()
print(f"\nFlat ChromaDB metadata sample:")
for k, v in flat.items():
    print(f"  {k}: {repr(v)}")

✓ models.py written and imported successfully

Flat ChromaDB metadata sample:
  chunk_id: 'test_001'
  condition: 'choking'
  condition_tags: 'airway|obstruction'
  urgency_level: 5
  age_group: 'adult'
  language: 'en'
  source: 'WHO_BEC_2016'
  step_type: 'action'
  section: 'Choking Treatment'
  page_ref: None
  keywords: 'choking|heimlich'


In [5]:
# Cell 4 — Write rag/seed_protocols.py
# Hand-verified chunks from WHO BEC 2016 + Nepal MoHP 2078
# English corpus — complete for 9 conditions

content = '''"""
NEVA RAG — Hand-verified seed protocol chunks (English).
Source: WHO Basic Emergency Care (2016) + Nepal MoHP Standard Treatment Protocol (2078 BS).
These are the safety baseline. PDF pipeline augments these; never replaces them.
"""

from rag.models import ProtocolChunk, ChunkMetadata

SEED_CHUNKS: list[ProtocolChunk] = [

    # ══════════════════════════════════════════════════════════════════════
    # 1. CHOKING
    # ══════════════════════════════════════════════════════════════════════

    ProtocolChunk(
        metadata=ChunkMetadata(
            chunk_id       = "WHO_BEC_choking_adult_assessment_01",
            condition      = "choking",
            condition_tags = ["airway obstruction", "foreign body", "unable to breathe",
                              "gagging", "silent cough", "turning blue", "cannot speak"],
            urgency_level  = 5,
            age_group      = "adult",
            language       = "en",
            source         = "WHO_BEC_2016",
            step_type      = "assessment",
            section        = "Choking — Recognition",
            keywords       = ["choking", "airway", "obstruction", "cough", "silent", "speak"],
        ),
        text=(
            "CHOKING — RECOGNITION (Adult):\\n"
            "MILD obstruction: person CAN cough forcefully, speak, or breathe. "
            "Encourage them to keep coughing. Do NOT intervene physically.\\n\\n"
            "SEVERE obstruction: person CANNOT cough effectively, cannot speak, cannot breathe, "
            "or makes a high-pitched noise while trying to inhale. "
            "They may clutch their throat with both hands (universal choking sign). "
            "Skin or lips may turn blue (cyanosis). "
            "This is an IMMEDIATE LIFE THREAT. Act within seconds."
        ),
    ),

    ProtocolChunk(
        metadata=ChunkMetadata(
            chunk_id       = "WHO_BEC_choking_adult_action_01",
            condition      = "choking",
            condition_tags = ["airway obstruction", "heimlich", "abdominal thrusts", "back blows"],
            urgency_level  = 5,
            age_group      = "adult",
            language       = "en",
            source         = "WHO_BEC_2016",
            step_type      = "action",
            section        = "Choking — Treatment (Conscious Adult)",
            keywords       = ["back blows", "abdominal thrusts", "heimlich", "choking"],
        ),
        text=(
            "CHOKING — TREATMENT (Conscious Adult):\\n"
            "Step 1 — Give 5 firm BACK BLOWS: stand to their side, "
            "lean them forward, support the chest with one hand, "
            "strike firmly between the shoulder blades with the heel of your other hand.\\n"
            "Step 2 — Check mouth after each blow. Remove object only if clearly visible.\\n"
            "Step 3 — If object not cleared: give 5 ABDOMINAL THRUSTS (Heimlich): "
            "stand behind them, make a fist just above the navel, "
            "grasp with your other hand, thrust sharply INWARD and UPWARD.\\n"
            "Step 4 — Alternate 5 back blows + 5 abdominal thrusts continuously.\\n"
            "Step 5 — If person becomes UNCONSCIOUS: lower them to ground carefully. "
            "Begin CPR. Before each rescue breath, look in mouth — "
            "remove object ONLY if you can clearly see it. Never do blind finger sweeps."
        ),
    ),

    ProtocolChunk(
        metadata=ChunkMetadata(
            chunk_id       = "WHO_BEC_choking_paediatric_action_01",
            condition      = "choking",
            condition_tags = ["choking infant", "choking child", "baby choking", "back blows infant"],
            urgency_level  = 5,
            age_group      = "paediatric",
            language       = "en",
            source         = "WHO_BEC_2016",
            step_type      = "action",
            section        = "Choking — Treatment (Infant under 1 year)",
            keywords       = ["choking", "infant", "baby", "child", "back blows", "chest thrusts"],
        ),
        text=(
            "CHOKING — INFANT UNDER 1 YEAR:\\n"
            "DO NOT use abdominal thrusts on infants — risk of organ injury.\\n"
            "Step 1 — Hold infant face-DOWN along your forearm, head lower than chest. "
            "Support head. Give 5 firm BACK BLOWS between shoulder blades with heel of hand.\\n"
            "Step 2 — Turn infant face-UP on your other forearm. "
            "Give 5 CHEST THRUSTS: two fingers on centre of chest, just below nipple line. "
            "Push down about 1.5 cm.\\n"
            "Step 3 — Check mouth after each sequence. Remove object only if clearly visible.\\n"
            "Step 4 — Alternate 5 back blows + 5 chest thrusts until object clears or infant loses consciousness.\\n"
            "Step 5 — If unconscious: begin infant CPR. Call emergency services immediately."
        ),
    ),

    ProtocolChunk(
        metadata=ChunkMetadata(
            chunk_id       = "WHO_BEC_choking_both_warning_01",
            condition      = "choking",
            condition_tags = ["choking warning", "blind sweep", "do not"],
            urgency_level  = 5,
            age_group      = "both",
            language       = "en",
            source         = "WHO_BEC_2016",
            step_type      = "warning",
            section        = "Choking — Critical Warnings",
            keywords       = ["blind sweep", "choking warning", "pregnant", "infant"],
        ),
        text=(
            "CHOKING — CRITICAL WARNINGS:\\n"
            "DO NOT perform blind finger sweeps in the mouth — pushes object deeper.\\n"
            "DO NOT use abdominal thrusts on infants under 1 year.\\n"
            "DO NOT use abdominal thrusts on pregnant women — use CHEST THRUSTS instead: "
            "hands on centre of chest, same technique as CPR compressions.\\n"
            "DO NOT hold person upside down and shake them.\\n"
            "DO NOT slap on the back while they are upright — lean them forward first."
        ),
    ),

    # ══════════════════════════════════════════════════════════════════════
    # 2. SEVERE BLEEDING
    # ══════════════════════════════════════════════════════════════════════

    ProtocolChunk(
        metadata=ChunkMetadata(
            chunk_id       = "WHO_BEC_bleeding_both_action_01",
            condition      = "severe_bleeding",
            condition_tags = ["haemorrhage", "hemorrhage", "blood loss", "wound",
                              "cut", "laceration", "stabbing", "gash", "injury bleeding"],
            urgency_level  = 5,
            age_group      = "both",
            language       = "en",
            source         = "WHO_BEC_2016",
            step_type      = "action",
            section        = "Bleeding — External Haemorrhage Control",
            keywords       = ["bleeding", "blood", "pressure", "tourniquet", "wound"],
        ),
        text=(
            "SEVERE EXTERNAL BLEEDING — IMMEDIATE CONTROL:\\n"
            "Step 1 — DIRECT PRESSURE: press a clean cloth or clothing firmly over the wound. "
            "Do NOT lift to check — keep pressing continuously for at least 10 minutes.\\n"
            "Step 2 — If blood soaks through: ADD more cloth on top. Do NOT remove the first cloth.\\n"
            "Step 3 — TOURNIQUET (limb wounds only, if direct pressure fails): "
            "apply a band 5–7 cm above the wound. Tighten until bleeding STOPS. "
            "Write the time of application on the person\\'s skin. "
            "Do NOT remove — hospital staff must remove it.\\n"
            "Step 4 — Lay person flat. Raise the bleeding limb above heart level "
            "IF no bone injury is suspected.\\n"
            "Step 5 — Keep warm. Monitor breathing. Do NOT give food or water.\\n"
            "Step 6 — Call emergency services. State \\'severe bleeding.\\'"
        ),
    ),

    ProtocolChunk(
        metadata=ChunkMetadata(
            chunk_id       = "WHO_BEC_bleeding_both_warning_01",
            condition      = "severe_bleeding",
            condition_tags = ["bleeding warning", "embedded object", "internal bleeding", "tourniquet warning"],
            urgency_level  = 5,
            age_group      = "both",
            language       = "en",
            source         = "WHO_BEC_2016",
            step_type      = "warning",
            section        = "Bleeding — Warnings",
            keywords       = ["tourniquet warning", "embedded object", "internal bleeding"],
        ),
        text=(
            "BLEEDING — CRITICAL WARNINGS:\\n"
            "DO NOT remove an embedded object from a wound — "
            "stabilise it in place and apply pressure AROUND it.\\n"
            "DO NOT remove a tourniquet once applied — "
            "removal can cause fatal sudden blood pressure drop.\\n"
            "DO NOT apply tourniquet over a joint (knee or elbow).\\n"
            "INTERNAL BLEEDING — suspect if: abdomen is hard or painful, "
            "blood from ears or nose without head injury, "
            "large bruising on abdomen or chest after trauma, "
            "person in shock (pale, cold, clammy, rapid weak pulse) with no visible wound. "
            "Action: lay flat, do NOT give food or water, immediate hospital, keep warm."
        ),
    ),

    # ══════════════════════════════════════════════════════════════════════
    # 3. UNCONSCIOUS / CARDIAC ARREST
    # ══════════════════════════════════════════════════════════════════════

    ProtocolChunk(
        metadata=ChunkMetadata(
            chunk_id       = "WHO_BEC_unconscious_adult_action_01",
            condition      = "unconscious",
            condition_tags = ["unresponsive", "collapsed", "not breathing", "no pulse",
                              "cardiac arrest", "CPR needed", "fainted", "coma"],
            urgency_level  = 5,
            age_group      = "adult",
            language       = "en",
            source         = "WHO_BEC_2016",
            step_type      = "action",
            section        = "Unresponsive Adult — Initial Management and CPR",
            keywords       = ["unconscious", "unresponsive", "CPR", "airway", "recovery position"],
        ),
        text=(
            "UNCONSCIOUS ADULT — STEP-BY-STEP:\\n"
            "Step 1 — CHECK RESPONSE: tap shoulders firmly, shout \\'Are you okay?\\'\\n"
            "Step 2 — If no response: SHOUT FOR HELP. Call 102 immediately.\\n"
            "Step 3 — OPEN AIRWAY: tilt head back gently, lift chin up.\\n"
            "Step 4 — CHECK BREATHING: look, listen, feel for up to 10 seconds.\\n\\n"
            "IF NOT BREATHING (or only gasping):\\n"
            "Begin CPR immediately:\\n"
            "• Place heel of hand on centre of chest (lower half of breastbone).\\n"
            "• Place second hand on top. Keep arms straight.\\n"
            "• Push down 5–6 cm. Release fully. 30 compressions at 100–120 per minute.\\n"
            "• Give 2 rescue breaths: seal mouth, one breath until chest rises. "
            "If untrained or unwilling — do COMPRESSIONS ONLY at 100–120 per minute.\\n"
            "• Continue 30:2 cycle until help arrives, person wakes, or you are too exhausted.\\n\\n"
            "IF BREATHING but unconscious:\\n"
            "RECOVERY POSITION: roll onto side, top knee bent forward, "
            "head tilted back to keep airway open. Monitor breathing continuously."
        ),
    ),

    ProtocolChunk(
        metadata=ChunkMetadata(
            chunk_id       = "WHO_BEC_unconscious_paediatric_action_01",
            condition      = "unconscious",
            condition_tags = ["child CPR", "infant CPR", "paediatric cardiac arrest", "child not breathing"],
            urgency_level  = 5,
            age_group      = "paediatric",
            language       = "en",
            source         = "WHO_BEC_2016",
            step_type      = "action",
            section        = "Unresponsive Child / Infant — CPR",
            keywords       = ["child CPR", "infant CPR", "paediatric", "not breathing", "two fingers"],
        ),
        text=(
            "CPR — CHILD (1 year to puberty):\\n"
            "• 5 initial rescue breaths before starting compressions.\\n"
            "• 30 compressions: one or two hands, push down one-third of chest depth.\\n"
            "• 2 rescue breaths. Continue 30:2.\\n"
            "• Rate: 100–120 per minute.\\n\\n"
            "CPR — INFANT (under 1 year):\\n"
            "• 5 initial rescue breaths.\\n"
            "• 30 compressions: two fingers on centre of chest, just below nipple line. "
            "Push down one-third of chest depth (~4 cm).\\n"
            "• 2 gentle rescue breaths covering both mouth AND nose.\\n"
            "• Rate: 100–120 per minute.\\n\\n"
            "KEY DIFFERENCE from adult CPR: children are more likely to have a breathing cause "
            "for arrest — give the 5 initial breaths before compressions."
        ),
    ),

    # ══════════════════════════════════════════════════════════════════════
    # 4. SNAKEBITE (Nepal-specific)
    # ══════════════════════════════════════════════════════════════════════

    ProtocolChunk(
        metadata=ChunkMetadata(
            chunk_id       = "MoHP_snakebite_both_action_01",
            condition      = "snakebite",
            condition_tags = ["snake bite", "venomous", "krait", "cobra", "viper",
                              "Russell viper", "saanp", "saap tokeko", "snake venom"],
            urgency_level  = 5,
            age_group      = "both",
            language       = "en",
            source         = "Nepal_MoHP_2078",
            step_type      = "action",
            section        = "Snakebite — Immediate First Aid",
            keywords       = ["snakebite", "snake", "venom", "antivenom", "immobilise"],
        ),
        text=(
            "SNAKEBITE — IMMEDIATE FIRST AID (Nepal MoHP):\\n"
            "Step 1 — Move person away from snake. Do NOT attempt to catch or kill the snake.\\n"
            "Step 2 — Keep person CALM and as STILL as possible. "
            "Movement increases blood flow and spreads venom faster.\\n"
            "Step 3 — Immobilise the bitten limb as if it were a fracture. "
            "Splint if available. Keep limb BELOW the level of the heart.\\n"
            "Step 4 — Remove rings, watches, bracelets, and tight clothing from the bitten limb "
            "BEFORE swelling starts.\\n"
            "Step 5 — Mark the leading edge of any swelling with a pen. Write the time. "
            "Repeat every 15 minutes — this helps doctors track progression.\\n"
            "Step 6 — Note the EXACT TIME of the bite — critical for antivenom dosing.\\n"
            "Step 7 — TRANSPORT IMMEDIATELY to hospital with antivenom capability. "
            "In Nepal: Bir Hospital (Kathmandu), BP Koirala Institute (Dharan), "
            "Bheri Hospital (Nepalgunj), provincial hospitals.\\n"
            "ANTIVENOM is the ONLY definitive treatment. First aid ONLY buys time."
        ),
    ),

    ProtocolChunk(
        metadata=ChunkMetadata(
            chunk_id       = "MoHP_snakebite_both_warning_01",
            condition      = "snakebite",
            condition_tags = ["snakebite do not", "tourniquet snake", "cut wound", "suck venom"],
            urgency_level  = 5,
            age_group      = "both",
            language       = "en",
            source         = "Nepal_MoHP_2078",
            step_type      = "warning",
            section        = "Snakebite — Critical Don'ts",
            keywords       = ["snakebite warning", "do not cut", "do not suck", "no tourniquet"],
        ),
        text=(
            "SNAKEBITE — CRITICAL WARNINGS (traditional remedies that KILL):\\n"
            "DO NOT cut or incise the bite wound.\\n"
            "DO NOT suck out venom — by mouth or any device.\\n"
            "DO NOT apply a tourniquet or tight bandage — causes tissue death and limb loss.\\n"
            "DO NOT apply ice, cold water, or heat to the wound.\\n"
            "DO NOT apply electric shock.\\n"
            "DO NOT apply herbs, traditional medicine, or any substance to the wound.\\n"
            "DO NOT give alcohol in any form.\\n"
            "DO NOT give Aspirin or Ibuprofen — "
            "these increase bleeding in haemotoxic envenomation (Russell\\'s viper, common in Nepal).\\n"
            "Paracetamol for pain is acceptable IF person is fully conscious and can swallow safely."
        ),
    ),

    ProtocolChunk(
        metadata=ChunkMetadata(
            chunk_id       = "MoHP_snakebite_both_assessment_01",
            condition      = "snakebite",
            condition_tags = ["snakebite symptoms", "venom signs", "neurotoxic", "haemotoxic", "local swelling"],
            urgency_level  = 5,
            age_group      = "both",
            language       = "en",
            source         = "Nepal_MoHP_2078",
            step_type      = "assessment",
            section        = "Snakebite — Envenomation Signs",
            keywords       = ["snakebite symptoms", "neurotoxic", "haemotoxic", "ptosis", "swelling"],
        ),
        text=(
            "SNAKEBITE — SIGNS OF SERIOUS ENVENOMATION (seek hospital URGENTLY):\\n\\n"
            "LOCAL SIGNS (any snake):\\n"
            "• Rapid and spreading swelling beyond the bite site\\n"
            "• Intense pain and redness spreading up the limb\\n"
            "• Blistering or tissue breakdown at bite site\\n\\n"
            "SYSTEMIC SIGNS — NEUROTOXIC (krait, cobra):\\n"
            "• Drooping eyelids (ptosis) — earliest sign\\n"
            "• Double vision, difficulty swallowing, slurred speech\\n"
            "• Weakness spreading to arms and legs\\n"
            "• Difficulty breathing — LIFE-THREATENING\\n\\n"
            "SYSTEMIC SIGNS — HAEMOTOXIC (Russell\\'s viper — most common fatal snake in Nepal):\\n"
            "• Bleeding from gums, injection sites, or old wounds\\n"
            "• Blood in urine (dark/brown urine)\\n"
            "• Swollen and tender lymph nodes\\n"
            "• Low blood pressure, rapid weak pulse\\n\\n"
            "ANY of the above = EMERGENCY. Hospital within 1 hour if possible."
        ),
    ),

    # ══════════════════════════════════════════════════════════════════════
    # 5. CHEST PAIN / SUSPECTED HEART ATTACK
    # ══════════════════════════════════════════════════════════════════════

    ProtocolChunk(
        metadata=ChunkMetadata(
            chunk_id       = "WHO_BEC_chest_pain_adult_assessment_01",
            condition      = "chest_pain",
            condition_tags = ["heart attack", "cardiac", "myocardial infarction",
                              "crushing chest", "jaw pain", "arm pain", "angina"],
            urgency_level  = 5,
            age_group      = "adult",
            language       = "en",
            source         = "WHO_BEC_2016",
            step_type      = "assessment",
            section        = "Chest Pain — Recognition of Cardiac Emergency",
            keywords       = ["chest pain", "heart attack", "cardiac", "crushing", "radiating"],
        ),
        text=(
            "CHEST PAIN — RECOGNISING A HEART ATTACK:\\n"
            "Classic signs:\\n"
            "• Crushing, squeezing, heavy pressure, or tightness in the chest\\n"
            "• Pain SPREADING to left arm, jaw, neck, back, or stomach\\n"
            "• Sweating (cold, clammy sweat) without exertion\\n"
            "• Nausea or vomiting\\n"
            "• Shortness of breath\\n"
            "• Pale or grey skin\\n"
            "• Sense of doom or extreme anxiety\\n\\n"
            "ATYPICAL SIGNS (more common in women, elderly, diabetics):\\n"
            "• Indigestion-like discomfort or stomach pain\\n"
            "• Unusual fatigue\\n"
            "• No chest pain — only breathlessness and sweating\\n\\n"
            "If ANY combination of the above: treat as heart attack. Do not wait to see if it improves."
        ),
    ),

    ProtocolChunk(
        metadata=ChunkMetadata(
            chunk_id       = "WHO_BEC_chest_pain_adult_action_01",
            condition      = "chest_pain",
            condition_tags = ["heart attack treatment", "aspirin cardiac", "nitrate", "cardiac first aid"],
            urgency_level  = 5,
            age_group      = "adult",
            language       = "en",
            source         = "WHO_BEC_2016",
            step_type      = "action",
            section        = "Chest Pain — First Aid Treatment",
            keywords       = ["chest pain", "aspirin", "cardiac first aid", "heart attack treatment"],
        ),
        text=(
            "CHEST PAIN — SUSPECTED HEART ATTACK FIRST AID:\\n"
            "Step 1 — STOP activity. Sit person down in the most comfortable position "
            "(usually half-sitting, knees bent). Do NOT allow them to walk.\\n"
            "Step 2 — Loosen tight clothing around neck and chest.\\n"
            "Step 3 — CALL 102 IMMEDIATELY. Say \\'I think this person is having a heart attack.\\' "
            "State your location clearly.\\n"
            "Step 4 — ASPIRIN: if the person is conscious, NOT allergic to aspirin, "
            "and can swallow — give 300 mg Aspirin. "
            "Ask them to CHEW it slowly, not swallow whole.\\n"
            "Step 5 — If person has doctor-prescribed nitrate spray (Sorbitrate/GTN): "
            "help them use it as prescribed.\\n"
            "Step 6 — Stay with them. Reassure them calmly. Keep monitoring.\\n"
            "Step 7 — If they lose consciousness and stop normal breathing: begin CPR immediately."
        ),
    ),

    # ══════════════════════════════════════════════════════════════════════
    # 6. BURNS
    # ══════════════════════════════════════════════════════════════════════

    ProtocolChunk(
        metadata=ChunkMetadata(
            chunk_id       = "WHO_BEC_burns_both_action_01",
            condition      = "burns",
            condition_tags = ["burn", "scald", "fire", "hot water", "boiling", "chemical burn", "electrical burn"],
            urgency_level  = 4,
            age_group      = "both",
            language       = "en",
            source         = "WHO_BEC_2016",
            step_type      = "action",
            section        = "Burns — First Aid",
            keywords       = ["burn", "scald", "cool water", "blister", "burn dressing"],
        ),
        text=(
            "BURNS — FIRST AID (4 Cs):\\n\\n"
            "1. COOL:\\n"
            "• Run COOL (not ice cold) running water over the burn for 20 MINUTES.\\n"
            "• Start within 3 hours of injury — still effective.\\n"
            "• Do NOT use ice or ice water — causes additional cold injury.\\n"
            "• For chemical burns: brush off any dry chemical FIRST, then irrigate with water for 20+ minutes.\\n"
            "• For electrical burns: DO NOT TOUCH person — disconnect power first, or use non-conducting object.\\n\\n"
            "2. CALL (when to get emergency help):\\n"
            "Call emergency or go to hospital immediately if:\\n"
            "• Burn larger than the person\\'s palm\\n"
            "• Burn on face, hands, feet, genitals, or over a joint\\n"
            "• Any burn in a child or elderly person\\n"
            "• Full-thickness burn: white, brown, or black skin; dry, leathery, painless at site\\n"
            "• Electrical or chemical burn (always hospital)\\n"
            "• Any burn with breathing difficulty (inhalation injury)\\n\\n"
            "3. COVER:\\n"
            "• Use clean cling film (best), clean plastic bag, or non-fluffy cloth.\\n"
            "• Do NOT use cotton wool, towels, or fluffy material — fibres stick to wound.\\n\\n"
            "4. COMFORT:\\n"
            "• Paracetamol for pain. Keep person warm (burns cause heat loss)."
        ),
    ),

    ProtocolChunk(
        metadata=ChunkMetadata(
            chunk_id       = "WHO_BEC_burns_both_warning_01",
            condition      = "burns",
            condition_tags = ["burn warning", "butter burn", "toothpaste burn", "blister pop"],
            urgency_level  = 4,
            age_group      = "both",
            language       = "en",
            source         = "WHO_BEC_2016",
            step_type      = "warning",
            section        = "Burns — Critical Warnings",
            keywords       = ["burn warning", "butter", "toothpaste", "ice burn", "blister"],
        ),
        text=(
            "BURNS — CRITICAL WARNINGS:\\n"
            "DO NOT apply butter, ghee, mustard oil, coconut oil, or any oil — "
            "traps heat and causes severe infection.\\n"
            "DO NOT apply toothpaste — causes infection and pain.\\n"
            "DO NOT apply egg white, raw potato, or any traditional remedy.\\n"
            "DO NOT pop or break blisters — blisters are the body\\'s sterile protective barrier.\\n"
            "DO NOT remove clothing stuck to burned skin — cut clothing around the stuck area.\\n"
            "DO NOT use ice or ice water — causes ice burns on top of heat burns.\\n"
            "DO NOT cover with cotton wool, towels, or fluffy bandages — fibres stick to raw wound."
        ),
    ),

    # ══════════════════════════════════════════════════════════════════════
    # 7. STROKE
    # ══════════════════════════════════════════════════════════════════════

    ProtocolChunk(
        metadata=ChunkMetadata(
            chunk_id       = "WHO_BEC_stroke_adult_assessment_01",
            condition      = "stroke",
            condition_tags = ["stroke FAST", "facial droop", "arm weakness", "speech slurred",
                              "brain attack", "sudden confusion", "sudden headache", "paralysis"],
            urgency_level  = 5,
            age_group      = "adult",
            language       = "en",
            source         = "WHO_BEC_2016",
            step_type      = "assessment",
            section        = "Stroke — FAST Recognition",
            keywords       = ["stroke", "FAST", "facial droop", "arm weakness", "speech"],
        ),
        text=(
            "STROKE — RECOGNITION (FAST Test):\\n\\n"
            "F — FACE: Ask person to SMILE. "
            "Does one side of the face droop? Is the smile uneven?\\n\\n"
            "A — ARMS: Ask person to RAISE BOTH ARMS. "
            "Does one arm drift downward or feel weak?\\n\\n"
            "S — SPEECH: Ask person to repeat a simple sentence. "
            "Is speech slurred, jumbled, or impossible?\\n\\n"
            "T — TIME: If ANY of the above is present — "
            "this is a STROKE. TIME IS BRAIN. Call emergency immediately.\\n\\n"
            "Additional warning signs:\\n"
            "• Sudden severe headache with no known cause (\\'worst headache of my life\\')\\n"
            "• Sudden vision loss in one or both eyes\\n"
            "• Sudden loss of balance or coordination\\n"
            "• Sudden numbness on one side of face, arm, or leg\\n\\n"
            "IMPORTANT: Symptoms that come and go (TIA / mini-stroke) are also emergencies — "
            "they often precede a major stroke within hours."
        ),
    ),

    ProtocolChunk(
        metadata=ChunkMetadata(
            chunk_id       = "WHO_BEC_stroke_adult_action_01",
            condition      = "stroke",
            condition_tags = ["stroke first aid", "brain attack treatment", "stroke management"],
            urgency_level  = 5,
            age_group      = "adult",
            language       = "en",
            source         = "WHO_BEC_2016",
            step_type      = "action",
            section        = "Stroke — First Aid",
            keywords       = ["stroke", "time is brain", "recovery position stroke", "no food stroke"],
        ),
        text=(
            "STROKE — FIRST AID:\\n"
            "Step 1 — CALL 102 IMMEDIATELY. "
            "State the time symptoms started — this is critical for hospital thrombolysis decision.\\n"
            "Step 2 — Note the EXACT time symptoms began and tell the hospital.\\n"
            "Step 3 — If conscious: sit or lay person in comfortable position. "
            "Support head and shoulders slightly raised.\\n"
            "Step 4 — If unconscious but breathing: RECOVERY POSITION (on their side).\\n"
            "Step 5 — DO NOT give food, water, or any medication — "
            "stroke affects swallowing and they may choke.\\n"
            "Step 6 — Do NOT leave person alone under any circumstances.\\n"
            "Step 7 — Keep them calm and reassured. Loosen tight clothing.\\n"
            "Step 8 — If they stop breathing: begin CPR.\\n\\n"
            "TIME IS BRAIN: Every 1 minute without treatment = 1.9 million brain cells lost. "
            "The clot-busting drug (tPA) must be given within 4.5 hours of symptom onset."
        ),
    ),

    # ══════════════════════════════════════════════════════════════════════
    # 8. ALTITUDE SICKNESS (Nepal-specific)
    # ══════════════════════════════════════════════════════════════════════

    ProtocolChunk(
        metadata=ChunkMetadata(
            chunk_id       = "MoHP_altitude_both_assessment_01",
            condition      = "altitude_sickness",
            condition_tags = ["AMS", "acute mountain sickness", "HACE", "HAPE",
                              "high altitude", "trekking", "Everest", "headache altitude", "Lake Louise"],
            urgency_level  = 4,
            age_group      = "both",
            language       = "en",
            source         = "Nepal_MoHP_2078",
            step_type      = "assessment",
            section        = "Altitude Sickness — Grading and Recognition",
            keywords       = ["altitude", "AMS", "HACE", "HAPE", "headache altitude", "ataxia"],
        ),
        text=(
            "ALTITUDE SICKNESS — RECOGNITION AND GRADING:\\n\\n"
            "MILD AMS (Lake Louise Score ≥3):\\n"
            "• Headache (the defining symptom)\\n"
            "• Fatigue, weakness\\n"
            "• Dizziness\\n"
            "• Nausea, loss of appetite\\n"
            "• Poor sleep\\n"
            "Onset: typically 6–12 hours after arriving at new altitude.\\n\\n"
            "SEVERE AMS / HACE (High Altitude Cerebral Oedema):\\n"
            "• Severe headache not relieved by paracetamol\\n"
            "• Confusion, disorientation, irrational behaviour\\n"
            "• Ataxia (cannot walk in a straight line — \\'walk the line test\\')\\n"
            "• Extreme fatigue\\n"
            "• Drowsiness progressing to unconsciousness — LIFE-THREATENING\\n\\n"
            "HAPE (High Altitude Pulmonary Oedema):\\n"
            "• Breathlessness at rest (not just on exertion)\\n"
            "• Dry cough progressing to cough with pink frothy sputum\\n"
            "• Cannot complete sentences without gasping\\n"
            "• Blue lips or fingernails (cyanosis) — IMMEDIATELY LIFE-THREATENING\\n\\n"
            "Ataxia test: ask person to walk heel-to-toe in a straight line. "
            "Failure = HACE until proven otherwise. Descend NOW."
        ),
    ),

    ProtocolChunk(
        metadata=ChunkMetadata(
            chunk_id       = "MoHP_altitude_both_action_01",
            condition      = "altitude_sickness",
            condition_tags = ["AMS treatment", "HACE treatment", "HAPE treatment",
                              "descend altitude", "Diamox", "Gamow bag", "Dexamethasone altitude"],
            urgency_level  = 4,
            age_group      = "both",
            language       = "en",
            source         = "Nepal_MoHP_2078",
            step_type      = "action",
            section        = "Altitude Sickness — Treatment by Severity",
            keywords       = ["altitude", "descend", "Dexamethasone", "Nifedipine", "Gamow", "Diamox"],
        ),
        text=(
            "ALTITUDE SICKNESS — TREATMENT:\\n\\n"
            "MILD AMS:\\n"
            "• STOP ascent. Rest at the SAME altitude for 24 hours.\\n"
            "• Hydrate well (3–4 litres water per day).\\n"
            "• Paracetamol 1g or Ibuprofen 400mg for headache.\\n"
            "• Acetazolamide (Diamox) 250mg twice daily if available — speeds acclimatisation.\\n"
            "• Do NOT ascend until COMPLETELY symptom-free.\\n\\n"
            "SEVERE AMS / HACE:\\n"
            "• DESCEND IMMEDIATELY — minimum 500–1000 m. Do not wait for morning.\\n"
            "• Dexamethasone 8 mg immediately (IM or oral), then 4 mg every 6 hours during descent.\\n"
            "• Supplemental oxygen if available (2–4 L/min).\\n"
            "• Gamow bag (portable hyperbaric chamber) if available — simulates descent.\\n\\n"
            "HAPE:\\n"
            "• IMMEDIATE DESCENT — highest priority.\\n"
            "• Oxygen 4–6 L/min.\\n"
            "• Nifedipine 30 mg slow-release if available.\\n"
            "• Gamow bag if descent not immediately possible.\\n\\n"
            "GOLDEN RULE: \\'If in doubt, descend.\\' Descent is the only definitive treatment."
        ),
    ),

    # ══════════════════════════════════════════════════════════════════════
    # 9. DROWNING
    # ══════════════════════════════════════════════════════════════════════

    ProtocolChunk(
        metadata=ChunkMetadata(
            chunk_id       = "WHO_BEC_drowning_both_action_01",
            condition      = "drowning",
            condition_tags = ["drowning", "near drowning", "water rescue", "submerged",
                              "pulled from water", "river", "swimming pool"],
            urgency_level  = 5,
            age_group      = "both",
            language       = "en",
            source         = "WHO_BEC_2016",
            step_type      = "action",
            section        = "Drowning — Rescue and Resuscitation",
            keywords       = ["drowning", "water", "rescue", "CPR", "secondary drowning"],
        ),
        text=(
            "DROWNING — RESCUE AND FIRST AID:\\n"
            "Step 1 — SAFE RESCUE: Do NOT enter fast or deep water unless trained.\\n"
            "• Throw: rope, clothing tied together, ring buoy, empty container.\\n"
            "• Reach: extend arm, stick, towel, or belt from the bank.\\n"
            "• Row: use a boat if available.\\n"
            "• Go in water ONLY as last resort, and only in shallow/calm water.\\n\\n"
            "Step 2 — Once out of water: CHECK RESPONSE (tap and shout).\\n"
            "Step 3 — If not breathing normally:\\n"
            "• Give 5 INITIAL RESCUE BREATHS immediately (before compressions).\\n"
            "• Drowning is a breathing emergency first — oxygen is the priority.\\n"
            "• Then begin 30:2 CPR. Continue until help arrives.\\n"
            "Step 4 — Do NOT waste time trying to drain water from lungs — it does not work.\\n"
            "Step 5 — Even if person REVIVES: mandatory hospital assessment.\\n"
            "• Secondary drowning: fluid in lungs can cause death 1–24 hours later.\\n"
            "• Watch for: persistent cough, breathing difficulty, unusual tiredness after rescue.\\n"
            "Step 6 — HYPOTHERMIA: remove wet clothing. Wrap in dry blanket. Keep warm."
        ),
    ),

    # ══════════════════════════════════════════════════════════════════════
    # 10. SHOCK
    # ══════════════════════════════════════════════════════════════════════

    ProtocolChunk(
        metadata=ChunkMetadata(
            chunk_id       = "WHO_BEC_shock_both_action_01",
            condition      = "shock",
            condition_tags = ["shock", "hypovolemic shock", "pale clammy", "rapid pulse",
                              "low blood pressure", "faint", "collapse after bleeding"],
            urgency_level  = 5,
            age_group      = "both",
            language       = "en",
            source         = "WHO_BEC_2016",
            step_type      = "action",
            section        = "Shock — Recognition and First Aid",
            keywords       = ["shock", "pale", "clammy", "rapid pulse", "lay flat", "legs raised"],
        ),
        text=(
            "SHOCK — RECOGNITION AND FIRST AID:\\n\\n"
            "RECOGNISE SHOCK:\\n"
            "• Pale, cold, clammy skin\\n"
            "• Rapid, weak pulse\\n"
            "• Rapid shallow breathing\\n"
            "• Confusion, restlessness, or unusual drowsiness\\n"
            "• Nausea\\n"
            "• Feeling faint or collapsing\\n\\n"
            "IMMEDIATE FIRST AID:\\n"
            "Step 1 — Lay person FLAT on their back.\\n"
            "Step 2 — Raise LEGS 30 cm above heart level (unless head, neck, spine, leg fracture, "
            "or breathing difficulty — then keep flat).\\n"
            "Step 3 — Treat the CAUSE if visible (control bleeding with direct pressure).\\n"
            "Step 4 — Keep WARM — cover with blanket.\\n"
            "Step 5 — DO NOT give food or water.\\n"
            "Step 6 — DO NOT leave person alone.\\n"
            "Step 7 — Call 102 immediately. State \\'person is in shock.\\' Monitor breathing constantly."
        ),
    ),

    # ══════════════════════════════════════════════════════════════════════
    # 11. SEIZURE
    # ══════════════════════════════════════════════════════════════════════

    ProtocolChunk(
        metadata=ChunkMetadata(
            chunk_id       = "WHO_BEC_seizure_both_action_01",
            condition      = "seizure",
            condition_tags = ["seizure", "epilepsy", "convulsion", "fit", "shaking", "tonic clonic"],
            urgency_level  = 4,
            age_group      = "both",
            language       = "en",
            source         = "WHO_BEC_2016",
            step_type      = "action",
            section        = "Seizure — First Aid",
            keywords       = ["seizure", "convulsion", "epilepsy", "fit", "protect head"],
        ),
        text=(
            "SEIZURE — FIRST AID:\\n"
            "Step 1 — PROTECT from injury: clear hard or sharp objects away. "
            "Cushion the head with something soft.\\n"
            "Step 2 — TIME the seizure from when it starts.\\n"
            "Step 3 — Do NOT restrain the person — you cannot stop a seizure by holding them.\\n"
            "Step 4 — Do NOT put ANYTHING in their mouth — "
            "people cannot swallow their tongue. Objects cause broken teeth and injury.\\n"
            "Step 5 — After convulsions stop: RECOVERY POSITION — roll onto side to protect airway.\\n"
            "Step 6 — Stay and monitor. Most seizures stop within 2–3 minutes. Person will be confused.\\n\\n"
            "CALL EMERGENCY if:\\n"
            "• Seizure lasts MORE than 5 minutes\\n"
            "• Another seizure follows without regaining consciousness\\n"
            "• Person does not wake up after seizure stops\\n"
            "• Person is injured during seizure\\n"
            "• Person is pregnant\\n"
            "• First-ever seizure\\n"
            "• Seizure in water"
        ),
    ),

]
'''

with open("/kaggle/working/neva/rag/seed_protocols.py", "w") as f:
    f.write(content)

# Verify
from rag.seed_protocols import SEED_CHUNKS
print(f"✓ seed_protocols.py written successfully")
print(f"✓ Total seed chunks: {len(SEED_CHUNKS)}")

# Summary by condition
from collections import Counter
condition_counts = Counter(c.metadata.condition for c in SEED_CHUNKS)
step_type_counts = Counter(c.metadata.step_type for c in SEED_CHUNKS)

print(f"\nChunks by condition:")
for cond, count in sorted(condition_counts.items()):
    print(f"  {cond:30s} {count}")

print(f"\nChunks by step_type:")
for stype, count in sorted(step_type_counts.items()):
    print(f"  {stype:20s} {count}")

✓ seed_protocols.py written successfully
✓ Total seed chunks: 22

Chunks by condition:
  altitude_sickness              2
  burns                          2
  chest_pain                     2
  choking                        4
  drowning                       1
  seizure                        1
  severe_bleeding                2
  shock                          1
  snakebite                      3
  stroke                         2
  unconscious                    2

Chunks by step_type:
  action               13
  assessment           5
  warning              4


In [6]:
# Cell 5 — Nepali mirror chunks for ALL 11 conditions

content = '''"""
NEVA RAG — Nepali (Devanagari) seed protocol chunks.
Mirrors of all 11 emergency conditions for full bilingual RAG support.
Sources: WHO Basic Emergency Care (2016) + Nepal MoHP Standard Treatment Protocol (2078 BS).
"""

from rag.models import ProtocolChunk, ChunkMetadata

SEED_CHUNKS_NE: list[ProtocolChunk] = [

    # ══════════════════════════════════════════════════════════════════════
    # 1. CHOKING (दम थिचिनु)
    # ══════════════════════════════════════════════════════════════════════

    ProtocolChunk(
        metadata=ChunkMetadata(
            chunk_id       = "WHO_BEC_choking_adult_action_01_ne",
            condition      = "choking",
            condition_tags = ["श्वासनली अवरोध", "दम थिचिनु", "सास लिन नसक्नु",
                              "नीलो हुनु", "अड्किनु", "घाँटीमा अड्कनु"],
            urgency_level  = 5,
            age_group      = "adult",
            language       = "ne",
            source         = "WHO_BEC_2016",
            step_type      = "action",
            section        = "दम थिचिनु — उपचार (सचेत वयस्क)",
            keywords       = ["दम थिचिनु", "श्वासनली", "ढाड थिच्नु", "पेट थिच्नु", "अड्किनु"],
        ),
        text=(
            "दम थिचिनु वा घाँटीमा अड्किनु — तुरुन्त गर्नुपर्ने काम (सचेत वयस्क):\\n"
            "पहिले जाँच गर्नुस्: बिरामी बोल्न वा खोक्न सक्छन् भने हल्का छ — खोक्न प्रोत्साहन दिनुस्।\\n"
            "बोल्न वा सास लिन नसकेमा (गम्भीर अवस्था):\\n"
            "चरण १ — ५ पटक ढाडमा बलियोसँग थिच्नुस्: बिरामीको छेउमा उभिनुस्, "
            "एक हातले छाती समाउनुस्, अर्को हातको हत्केलाले काँधका हाडहरूको बीचमा बलियोसँग थिच्नुस्।\\n"
            "चरण २ — हरेक थिचाइपछि मुख हेर्नुस्। वस्तु देखिएमा मात्र निकाल्नुस्।\\n"
            "चरण ३ — वस्तु नझरेमा ५ पटक पेट थिच्नुस् (Heimlich): "
            "पछाडिबाट उभिनुस्, नाइटोभन्दा माथि मुठ्ठी बनाउनुस्, "
            "अर्को हातले मुठ्ठी समाउनुस्, भित्र र माथितिर जोडले धकेल्नुस्।\\n"
            "चरण ४ — ५ ढाड थिचाइ र ५ पेट थिचाइ पालैपालो गर्दै रहनुस्।\\n"
            "चरण ५ — बिरामी बेहोस भएमा: बिस्तारै भुइँमा सुताउनुस्। "
            "तुरुन्त CPR सुरु गर्नुस् र १०२ मा फोन गर्नुस्।"
        ),
    ),

    ProtocolChunk(
        metadata=ChunkMetadata(
            chunk_id       = "WHO_BEC_choking_both_warning_01_ne",
            condition      = "choking",
            condition_tags = ["दम थिचिनु चेतावनी", "औंला नहाल्नुस्", "बच्चा दम थिचिनु"],
            urgency_level  = 5,
            age_group      = "both",
            language       = "ne",
            source         = "WHO_BEC_2016",
            step_type      = "warning",
            section        = "दम थिचिनु — गर्न नहुने काम",
            keywords       = ["दम थिचिनु", "चेतावनी", "औंला", "बच्चा", "गर्भवती"],
        ),
        text=(
            "दम थिचिनु — गर्न नहुने काम:\\n"
            "मुखमा औंला अन्धाधुन्ध नहाल्नुस् — यसले वस्तु झन् भित्र धकेलिन सक्छ।\\n"
            "१ वर्षभन्दा सानो बच्चालाई पेट कहिल्यै नथिच्नुस् — ढाड र छाती मात्र थिच्नुस्।\\n"
            "गर्भवती महिलालाई पेट नथिच्नुस् — छाती थिच्नुस्।\\n"
            "सिधा उभिएको अवस्थामा ढाड नहिर्काउनुस् — अगाडि झुकाएर मात्र थिच्नुस्।"
        ),
    ),

    # ══════════════════════════════════════════════════════════════════════
    # 2. SEVERE BLEEDING (धेरै रगत बग्नु)
    # ══════════════════════════════════════════════════════════════════════

    ProtocolChunk(
        metadata=ChunkMetadata(
            chunk_id       = "WHO_BEC_bleeding_both_action_01_ne",
            condition      = "severe_bleeding",
            condition_tags = ["धेरै रगत बग्नु", "घाउ", "चोट", "रगत रोक्नु",
                              "काटिनु", "रगत धेरै बगेको"],
            urgency_level  = 5,
            age_group      = "both",
            language       = "ne",
            source         = "WHO_BEC_2016",
            step_type      = "action",
            section        = "धेरै रगत बग्नु — तुरुन्त रगत रोक्ने तरिका",
            keywords       = ["रगत", "घाउ", "दबाउनु", "पट्टी", "ट्युर्निकेट"],
        ),
        text=(
            "धेरै रगत बग्नु — तुरुन्त गर्नुपर्ने काम:\\n"
            "चरण १ — सिधा दबाउनुस्: सफा कपडा वा लुगा घाउमाथि राखेर बलियोसँग थिच्नुस्। "
            "कम्तीमा १० मिनेट नरोकी थिच्नुस्। हेर्न कपडा नउठाउनुस्।\\n"
            "चरण २ — रगत सोकेमा: माथिबाट थप कपडा राख्नुस्। पहिलो कपडा नहटाउनुस्।\\n"
            "चरण ३ — हात वा खुट्टामा रगत नरोकेमा ट्युर्निकेट लगाउनुस्: "
            "घाउभन्दा ५–७ सेमि माथि कस्नुस्। रगत नरोकुन्जेल कस्दै जानुस्। "
            "लगाएको समय छालामा लेख्नुस्। ट्युर्निकेट आफैं कहिल्यै नहटाउनुस्।\\n"
            "चरण ४ — बिरामीलाई सुताउनुस्। हड्डी नभाँचिएको भए घाउ भएको अंग मुटुभन्दा माथि उठाउनुस्।\\n"
            "चरण ५ — तातो राख्नुस्। खाना वा पानी नदिनुस्। तुरुन्त १०२ मा फोन गर्नुस्।"
        ),
    ),

    ProtocolChunk(
        metadata=ChunkMetadata(
            chunk_id       = "WHO_BEC_bleeding_both_warning_01_ne",
            condition      = "severe_bleeding",
            condition_tags = ["रगत बग्नु चेतावनी", "भित्री रगत", "अड्किएको वस्तु"],
            urgency_level  = 5,
            age_group      = "both",
            language       = "ne",
            source         = "WHO_BEC_2016",
            step_type      = "warning",
            section        = "धेरै रगत बग्नु — गर्न नहुने काम",
            keywords       = ["रगत चेतावनी", "ट्युर्निकेट नहटाउनुस्", "भित्री रगत"],
        ),
        text=(
            "धेरै रगत बग्नु — गर्न नहुने काम:\\n"
            "घाउमा अड्किएको वस्तु नतान्नुस् — वरिपरि कपडाले थिचेर स्थिर राख्नुस्।\\n"
            "ट्युर्निकेट एकचोटि लगाइसकेपछि नहटाउनुस् — हटाउँदा अचानक रक्तचाप घटेर ज्यान जान सक्छ।\\n"
            "घुँडा वा कुहिनो जोर्नीमाथि ट्युर्निकेट नलगाउनुस्।\\n"
            "भित्री रगत बग्नुको लक्षण: पेट कडा वा दुखेको, कान वा नाकबाट रगत, पेटमा ठूलो निलडाम। "
            "यस्तोमा: सुताउनुस्, खान नदिनुस्, तुरुन्त अस्पताल।"
        ),
    ),

    # ══════════════════════════════════════════════════════════════════════
    # 3. UNCONSCIOUS / CPR (बेहोस)
    # ══════════════════════════════════════════════════════════════════════

    ProtocolChunk(
        metadata=ChunkMetadata(
            chunk_id       = "WHO_BEC_unconscious_adult_action_01_ne",
            condition      = "unconscious",
            condition_tags = ["बेहोस", "सास नचल्नु", "ढलेको", "होस नहुनु",
                              "मृत्यु प्राय", "जवाफ नदिनु"],
            urgency_level  = 5,
            age_group      = "adult",
            language       = "ne",
            source         = "WHO_BEC_2016",
            step_type      = "action",
            section        = "बेहोस वयस्क — CPR र प्रारम्भिक उपचार",
            keywords       = ["बेहोस", "CPR", "छाती थिच्नु", "श्वासनली"],
        ),
        text=(
            "बेहोस व्यक्ति — तुरुन्त गर्नुपर्ने काम:\\n"
            "चरण १ — प्रतिक्रिया जाँच गर्नुस्: काँध थिचेर जोडले बोलाउनुस् \\'ठीक हुनुहुन्छ?\\'\\n"
            "चरण २ — होस नभएमा: मद्दत माग्नुस्। तुरुन्त १०२ मा फोन गर्नुस्।\\n"
            "चरण ३ — श्वासनली खोल्नुस्: टाउको पछाडि झुकाउनुस्, चिउँडो माथि उठाउनुस्।\\n"
            "चरण ४ — सास जाँच गर्नुस्: हेर्नुस्, सुन्नुस्, महसुस गर्नुस् — १० सेकेन्डसम्म।\\n\\n"
            "सास नचलेमा (CPR तुरुन्त सुरु गर्नुस्):\\n"
            "• छातीको बीचमा (स्टर्नमको तल्लो आधा) हात राख्नुस्।\\n"
            "• सिधा हातले ५–६ सेमि गहिरो थिच्नुस्। पूरा छोड्नुस्।\\n"
            "• प्रति मिनेट १००–१२० पटकका दरले ३० पटक थिच्नुस्।\\n"
            "• २ पटक उद्धार श्वास दिनुस् — मुख बन्द गरी छाती उठेसम्म फुक्नुस्।\\n"
            "• प्रशिक्षण नभए केवल छाती मात्र थिच्नुस्, नरोकी।\\n\\n"
            "सास चलेको तर बेहोस भएमा:\\n"
            "• कोल्टे (Recovery Position) सुताउनुस्: छेउमा पल्टाउनुस्, "
            "टाउको पछाडि झुकाउनुस्। निरन्तर जाँच गर्दै रहनुस्।"
        ),
    ),

    ProtocolChunk(
        metadata=ChunkMetadata(
            chunk_id       = "WHO_BEC_unconscious_paediatric_action_01_ne",
            condition      = "unconscious",
            condition_tags = ["बच्चा बेहोस", "बच्चा CPR", "शिशु CPR", "बच्चा सास नचल्नु"],
            urgency_level  = 5,
            age_group      = "paediatric",
            language       = "ne",
            source         = "WHO_BEC_2016",
            step_type      = "action",
            section        = "बच्चा र शिशु CPR",
            keywords       = ["बच्चा CPR", "शिशु CPR", "दुई औंला", "सास दिनु"],
        ),
        text=(
            "CPR — बच्चा (१ वर्षदेखि किशोरावस्थासम्म):\\n"
            "• छाती थिच्नुअघि ५ पटक सास दिनुस्।\\n"
            "• ३० पटक छाती थिच्नुस्: एक वा दुई हातले, छातीको एक तिहाइ गहिराइसम्म।\\n"
            "• २ पटक सास दिनुस्। प्रति मिनेट १००–१२० पटक।\\n\\n"
            "CPR — शिशु (१ वर्षभन्दा सानो):\\n"
            "• ५ पटक सास दिनुस् (मुख र नाक दुवै ढाकेर)।\\n"
            "• ३० पटक छाती थिच्नुस्: दुई औंलाले स्तनको ठिक तल, "
            "छातीको एक तिहाइ गहिराइसम्म (करिब ४ सेमि)।\\n"
            "• २ पटक सास दिनुस्। प्रति मिनेट १००–१२० पटक।\\n\\n"
            "महत्त्वपूर्ण: बच्चाको श्वास नै मुख्य समस्या हुन्छ — त्यसैले "
            "थिच्नुअघि ५ पटक सास दिनु अनिवार्य छ।"
        ),
    ),

    # ══════════════════════════════════════════════════════════════════════
    # 4. SNAKEBITE (साँप टोकेको)
    # ══════════════════════════════════════════════════════════════════════

    ProtocolChunk(
        metadata=ChunkMetadata(
            chunk_id       = "MoHP_snakebite_both_action_01_ne",
            condition      = "snakebite",
            condition_tags = ["साँप टोकेको", "सर्प दंश", "विषालु साँप",
                              "साँप", "विष", "सर्पदंश"],
            urgency_level  = 5,
            age_group      = "both",
            language       = "ne",
            source         = "Nepal_MoHP_2078",
            step_type      = "action",
            section        = "साँप टोकेको — तुरुन्त प्राथमिक उपचार",
            keywords       = ["साँप", "विष", "एन्टिभेनम", "स्थिर राख्नु", "अस्पताल"],
        ),
        text=(
            "साँप टोकेको — तुरुन्त गर्नुपर्ने काम (नेपाल MoHP):\\n"
            "चरण १ — बिरामीलाई साँपबाट टाढा लैजानुस्। साँप समाउन वा मार्न नजानुस्।\\n"
            "चरण २ — बिरामीलाई शान्त र स्थिर राख्नुस्। "
            "हिँड्दा वा डराउँदा विष छिटो फैलिन्छ।\\n"
            "चरण ३ — टोकिएको अंग हड्डी भाँचिएझैँ स्थिर राख्नुस्। "
            "स्प्लिन्ट लगाउन सकिन्छ। अंग मुटुभन्दा तल राख्नुस्।\\n"
            "चरण ४ — टोकिएको अंगका औंठी, घडी, कडा पट्टी वा तंग कपडा "
            "सुन्निनुअघि नै हटाउनुस्।\\n"
            "चरण ५ — सुन्निएको किनारमा कलमले रेखा कोर्नुस् र समय लेख्नुस्। "
            "हरेक १५ मिनेटमा दोहोर्याउनुस्।\\n"
            "चरण ६ — टोकिएको सही समय नोट गर्नुस्।\\n"
            "चरण ७ — तुरुन्त एन्टिभेनम भएको अस्पताल जानुस्: "
            "वीर अस्पताल (काठमाडौँ), बीपी कोइराला (धरान), भेरी अस्पताल (नेपालगञ्ज)।\\n"
            "एन्टिभेनम नै एकमात्र सही उपचार हो। प्राथमिक उपचारले समय मात्र बचाउँछ।"
        ),
    ),

    ProtocolChunk(
        metadata=ChunkMetadata(
            chunk_id       = "MoHP_snakebite_both_warning_01_ne",
            condition      = "snakebite",
            condition_tags = ["साँप टोकेको चेतावनी", "घाउ काट्नु हुँदैन",
                              "विष चुस्नु हुँदैन", "परम्परागत उपचार"],
            urgency_level  = 5,
            age_group      = "both",
            language       = "ne",
            source         = "Nepal_MoHP_2078",
            step_type      = "warning",
            section        = "साँप टोकेको — गर्न नहुने काम",
            keywords       = ["साँप चेतावनी", "घाउ काट्नु", "विष चुस्नु", "ट्युर्निकेट साँप"],
        ),
        text=(
            "साँप टोकेको — यी काम गर्नु हुँदैन (परम्परागत उपचारले मृत्यु हुन सक्छ):\\n"
            "घाउमा काट्नु वा चिर्नु हुँदैन।\\n"
            "मुखले वा कुनै यन्त्रले विष चुस्नु हुँदैन।\\n"
            "ट्युर्निकेट वा कडा पट्टी लगाउनु हुँदैन — मासु कुहिन्छ र अंग गुम्न सक्छ।\\n"
            "बरफ, चिसो पानी वा तातो कुरा घाउमा लगाउनु हुँदैन।\\n"
            "विद्युत्को झड्का दिनु हुँदैन।\\n"
            "जडीबुटी, धामी-झाँक्री उपचार वा कुनै पनि परम्परागत औषधि लगाउनु हुँदैन।\\n"
            "रक्सी दिनु हुँदैन।\\n"
            "एस्पिरिन वा इबुप्रोफेन दिनु हुँदैन — रगत पातलो बनाउँछ।\\n"
            "पूर्ण होस भएको र निल्न सक्ने व्यक्तिलाई मात्र पेरासिटामल दिन मिल्छ।"
        ),
    ),

    # ══════════════════════════════════════════════════════════════════════
    # 5. CHEST PAIN (छाती दुख्नु / हृदयघात)
    # ══════════════════════════════════════════════════════════════════════

    ProtocolChunk(
        metadata=ChunkMetadata(
            chunk_id       = "WHO_BEC_chest_pain_adult_assessment_01_ne",
            condition      = "chest_pain",
            condition_tags = ["छाती दुख्नु", "हृदयघात", "हार्ट अट्याक",
                              "मुटु दुख्नु", "छाती थिचिनु"],
            urgency_level  = 5,
            age_group      = "adult",
            language       = "ne",
            source         = "WHO_BEC_2016",
            step_type      = "assessment",
            section        = "हृदयघात — पहिचान",
            keywords       = ["हृदयघात", "छाती दुख्नु", "देब्रे हात", "पसिना"],
        ),
        text=(
            "हृदयघात — पहिचान:\\n"
            "यी लक्षण देखिए हृदयघात हुन सक्छ:\\n"
            "• छाती भारी, थिचिएको, मुच्चिएको वा जलेको जस्तो अनुभव।\\n"
            "• देब्रे हात, बङ्गारा, घाँटी, ढाड वा पेटतिर दुखाइ फैलिनु।\\n"
            "• बिना कारण चिसो पसिना आउनु।\\n"
            "• वाकवाकी वा बान्ता।\\n"
            "• सास फेर्न गाह्रो हुनु।\\n"
            "• छाला फिक्का वा खरानी रंगको हुनु।\\n"
            "• अत्यन्त चिन्ता वा केही अनिष्ट हुने अनुभव।\\n\\n"
            "महिला, बृद्ध र मधुमेह भएकाहरूमा छाती नदुखेर पेट दुख्ने, "
            "असामान्य थकान वा सास मात्र फुल्ने पनि हुन सक्छ।\\n"
            "माथिका कुनै पनि लक्षण देखिए: हृदयघात मानेर उपचार गर्नुस्। पर्खिनुस् नहोस्।"
        ),
    ),

    ProtocolChunk(
        metadata=ChunkMetadata(
            chunk_id       = "WHO_BEC_chest_pain_adult_action_01_ne",
            condition      = "chest_pain",
            condition_tags = ["हृदयघात उपचार", "एस्पिरिन", "मुटु पहिलो उपचार"],
            urgency_level  = 5,
            age_group      = "adult",
            language       = "ne",
            source         = "WHO_BEC_2016",
            step_type      = "action",
            section        = "हृदयघात — पहिलो उपचार",
            keywords       = ["हृदयघात", "एस्पिरिन", "CPR", "१०२"],
        ),
        text=(
            "छाती दुख्नु (हृदयघातको शंका) — तुरुन्त गर्नुपर्ने काम:\\n"
            "चरण १ — बिरामीलाई सजिलो गरी (आधा उठेर, खुट्टा अगाडि राखेर) बसाउनुस्। "
            "हिँडडुल गर्न पूर्ण रूपमा बन्द गर्नुस्।\\n"
            "चरण २ — घाँटी र छाती वरपरका कसिला कपडा खुकुलो पार्नुस्।\\n"
            "चरण ३ — तुरुन्त १०२ मा फोन गर्नुस्। \\'हृदयघात जस्तो देखिन्छ\\' भन्नुस् र ठाउँ स्पष्ट बताउनुस्।\\n"
            "चरण ४ — बिरामीको होस छ, एलर्जी छैन र निल्न सक्छन् भने "
            "३०० मिलिग्राम एस्पिरिन (Aspirin) चपाउन दिनुस् — निल्न होइन।\\n"
            "चरण ५ — डाक्टरले दिएको नाइट्रेट स्प्रे (Sorbitrate/GTN) छ भने सहयोग गर्नुस्।\\n"
            "चरण ६ — शान्त राख्नुस्। एक्लै नछोड्नुस्। निरन्तर जाँच गर्दै रहनुस्।\\n"
            "चरण ७ — बेहोस भई सास बन्द भएमा: तुरुन्त CPR सुरु गर्नुस्।"
        ),
    ),

    # ══════════════════════════════════════════════════════════════════════
    # 6. BURNS (पोल्नु)
    # ══════════════════════════════════════════════════════════════════════

    ProtocolChunk(
        metadata=ChunkMetadata(
            chunk_id       = "WHO_BEC_burns_both_action_01_ne",
            condition      = "burns",
            condition_tags = ["पोल्नु", "आगोले पोलेको", "तातो पानीले पोलेको",
                              "जल्नु", "आगो लाग्नु", "रासायनिक पोल्नु"],
            urgency_level  = 4,
            age_group      = "both",
            language       = "ne",
            source         = "WHO_BEC_2016",
            step_type      = "action",
            section        = "पोल्नु — प्राथमिक उपचार",
            keywords       = ["पोल्नु", "चिसो पानी", "छोप्नु", "अस्पताल"],
        ),
        text=(
            "पोल्नु — तुरुन्त गर्नुपर्ने काम (४ C):\\n\\n"
            "१. चिसो पानी (Cool):\\n"
            "• बगिरहेको चिसो (तर बरफ होइन) पानीले पोलेको ठाउँमा कम्तीमा २० मिनेट पखाल्नुस्।\\n"
            "• चोट लागेको ३ घण्टाभित्र पखाल्दा पनि फाइदा हुन्छ।\\n"
            "• रासायनिक पोल्दा: पहिले सुकेको रसायन हटाउनुस्, त्यसपछि २०+ मिनेट पानी लगाउनुस्।\\n"
            "• बिजुलीले पोल्दा: पहिले बिजुली काट्नुस् वा नसंग्लने वस्तुले हटाउनुस्। बिरामीलाई हात नलगाउनुस्।\\n\\n"
            "२. अस्पताल कहिले जाने (Call):\\n"
            "• पोलेको भाग हत्केलाभन्दा ठूलो छ\\n"
            "• अनुहार, हात, खुट्टा, गुप्ताङ्ग वा जोर्नीमा छ\\n"
            "• बच्चा वा बृद्धलाई पोलेको छ\\n"
            "• छाला सेतो, खैरो वा कालो भएको छ र दुखाइ नभएको छ (गहिरो पोल्नु)\\n"
            "• बिजुली वा रसायनले पोलेको छ\\n"
            "• सास फेर्न गाह्रो छ\\n\\n"
            "३. छोप्नु (Cover):\\n"
            "• सफा प्लास्टिक (Cling film) वा सफा थैलोले हल्का छोप्नुस्।\\n"
            "• रुघा, तौलिया वा भुवाले नछोप्नुस् — तन्तु घाउमा टाँसिन्छ।\\n\\n"
            "४. आराम (Comfort):\\n"
            "• पेरासिटामल दिन सकिन्छ। तातो रहन सहयोग गर्नुस्।"
        ),
    ),

    ProtocolChunk(
        metadata=ChunkMetadata(
            chunk_id       = "WHO_BEC_burns_both_warning_01_ne",
            condition      = "burns",
            condition_tags = ["पोल्नु चेतावनी", "घ्यू नलगाउनुस्", "टुथपेस्ट",
                              "फोका नफुटाउनुस्", "बरफ नलगाउनुस्"],
            urgency_level  = 4,
            age_group      = "both",
            language       = "ne",
            source         = "WHO_BEC_2016",
            step_type      = "warning",
            section        = "पोल्नु — गर्न नहुने काम",
            keywords       = ["पोल्नु चेतावनी", "घ्यू", "टुथपेस्ट", "फोका", "बरफ"],
        ),
        text=(
            "पोल्नु — गर्न नहुने काम:\\n"
            "घ्यू, तेल, नरिवल तेल, सरसोको तेल वा कुनै पनि तेल नलगाउनुस् — "
            "तातो थुनिन्छ र संक्रमण हुन्छ।\\n"
            "टुथपेस्ट नलगाउनुस् — संक्रमण र दुखाइ बढाउँछ।\\n"
            "अण्डाको सेतो भाग, काँचो आलु वा कुनै पनि परम्परागत लेप नलगाउनुस्।\\n"
            "फोका नफुटाउनुस् — शरीरको आफ्नै संक्रमण-रोधी ढाल हो।\\n"
            "घाउमा टाँसिएको कपडा नतान्नुस् — वरिपरिबाट काट्नुस्।\\n"
            "बरफ वा अत्यन्त चिसो पानी नलगाउनुस् — थप चोट पर्छ।\\n"
            "भुवा (cotton wool), रुघा वा तौलिया नलगाउनुस् — तन्तु घाउमा टाँसिन्छ।"
        ),
    ),

    # ══════════════════════════════════════════════════════════════════════
    # 7. STROKE (पक्षघात / मस्तिष्काघात)
    # ══════════════════════════════════════════════════════════════════════

    ProtocolChunk(
        metadata=ChunkMetadata(
            chunk_id       = "WHO_BEC_stroke_adult_assessment_01_ne",
            condition      = "stroke",
            condition_tags = ["पक्षघात", "मस्तिष्काघात", "स्ट्रोक",
                              "मुख बाङ्गिनु", "हात काम नगर्नु", "बोली लरबराउनु"],
            urgency_level  = 5,
            age_group      = "adult",
            language       = "ne",
            source         = "WHO_BEC_2016",
            step_type      = "assessment",
            section        = "पक्षघात — FAST जाँच",
            keywords       = ["पक्षघात", "FAST", "मुख बाङ्गिनु", "हात", "बोली"],
        ),
        text=(
            "पक्षघात (Stroke) — FAST जाँच:\\n\\n"
            "F (Face — अनुहार): हाँस्न लगाउनुस्। "
            "अनुहारको एक भाग बाङ्गिएको वा झरेको छ?\\n\\n"
            "A (Arms — हात): दुवै हात उठाउन लगाउनुस्। "
            "एउटा हात आफैं तल झर्छ वा कमजोर छ?\\n\\n"
            "S (Speech — बोली): एउटा सजिलो वाक्य दोहोर्याउन लगाउनुस्। "
            "बोली लरबराएको, अड्किएको वा अर्थहीन छ?\\n\\n"
            "T (Time — समय): माथिका कुनै पनि लक्षण छन् भने "
            "तुरुन्त १०२ मा फोन गर्नुस् — एक मिनेट ढिला हुँदा १९ लाख मस्तिष्क कोशिका नष्ट हुन्छन्।\\n\\n"
            "थप लक्षण: अचानक कडा टाउको दुख्नु, एउटा आँखाको दृष्टि जानु, "
            "सन्तुलन गुम्नु, शरीरको एक भागमा सुन्निनु।"
        ),
    ),

    ProtocolChunk(
        metadata=ChunkMetadata(
            chunk_id       = "WHO_BEC_stroke_adult_action_01_ne",
            condition      = "stroke",
            condition_tags = ["पक्षघात उपचार", "स्ट्रोक उपचार", "मस्तिष्काघात"],
            urgency_level  = 5,
            age_group      = "adult",
            language       = "ne",
            source         = "WHO_BEC_2016",
            step_type      = "action",
            section        = "पक्षघात — प्राथमिक उपचार",
            keywords       = ["पक्षघात", "समय", "खान नदिनुस्", "CPR"],
        ),
        text=(
            "पक्षघात — तुरुन्त गर्नुपर्ने काम:\\n"
            "चरण १ — तुरुन्त १०२ मा फोन गर्नुस्। \\'पक्षघात जस्तो देखिन्छ\\' भन्नुस्।\\n"
            "चरण २ — लक्षण सुरु भएको ठ्याक्कै समय याद राख्नुस् — "
            "यो डाक्टरले औषधि दिन सक्ने वा नसक्ने निर्धारण गर्छ।\\n"
            "चरण ३ — होस छ भने: सजिलो गरी बसाउनुस् वा टाउको र काँध अलि उठाएर सुताउनुस्।\\n"
            "चरण ४ — बेहोस तर सास फेरेको छ भने: कोल्टे सुताउनुस्।\\n"
            "चरण ५ — खान वा पिउन केही पनि नदिनुस् — पक्षघातले निल्ने क्षमता गुमाउँछ र "
            "सर्किएर ज्यान जान सक्छ।\\n"
            "चरण ६ — एक्लै कहिल्यै नछोड्नुस्। शान्त राख्नुस्।\\n"
            "चरण ७ — सास बन्द भएमा: तुरुन्त CPR सुरु गर्नुस्।"
        ),
    ),

    # ══════════════════════════════════════════════════════════════════════
    # 8. ALTITUDE SICKNESS (लेक लाग्नु)
    # ══════════════════════════════════════════════════════════════════════

    ProtocolChunk(
        metadata=ChunkMetadata(
            chunk_id       = "MoHP_altitude_both_assessment_01_ne",
            condition      = "altitude_sickness",
            condition_tags = ["लेक लाग्नु", "AMS", "HACE", "HAPE",
                              "उचाइमा बिरामी", "टाउको दुख्ने उचाइ"],
            urgency_level  = 4,
            age_group      = "both",
            language       = "ne",
            source         = "Nepal_MoHP_2078",
            step_type      = "assessment",
            section        = "लेक लाग्नु — पहिचान र गम्भीरता",
            keywords       = ["लेक", "टाउको दुख्नु", "HACE", "HAPE", "सिधा हिँड्न नसक्नु"],
        ),
        text=(
            "लेक लाग्नु — पहिचान र गम्भीरता:\\n\\n"
            "सामान्य लेक लागेको (AMS):\\n"
            "• टाउको दुख्नु (मुख्य लक्षण), थकान, रिङ्गटा, वाकवाकी, निद्रा नलाग्नु।\\n"
            "• नयाँ उचाइमा पुगेको ६–१२ घण्टापछि सुरु हुन्छ।\\n\\n"
            "गम्भीर लेक (HACE — मस्तिष्कमा पानी जम्नु):\\n"
            "• कडा टाउको दुख्नु जुन सिटामोलले पनि कम नहुने।\\n"
            "• बरबराउनु, भ्रम हुनु, अस्वाभाविक व्यवहार।\\n"
            "• सिधा रेखामा हिँड्न नसक्नु (यो सबैभन्दा महत्त्वपूर्ण जाँच हो — ज्यानलाई खतरा)।\\n"
            "• अत्यधिक थकान वा बेहोस हुनु।\\n\\n"
            "फोक्सोमा पानी जम्नु (HAPE — अत्यन्त खतरनाक):\\n"
            "• आराम गर्दा पनि सास फुल्नु।\\n"
            "• खोक्दा गुलाबी वा रातो झागदार कफ आउनु।\\n"
            "• ओठ वा औंला नीलो हुनु।\\n\\n"
            "सिधा हिँड्न नसकेको देखे: तुरुन्त तल झार्नुस् — पर्खिनु जोखिमपूर्ण छ।"
        ),
    ),

    ProtocolChunk(
        metadata=ChunkMetadata(
            chunk_id       = "MoHP_altitude_both_action_01_ne",
            condition      = "altitude_sickness",
            condition_tags = ["लेक लाग्नु उपचार", "तल झर्नु", "Dexamethasone",
                              "Gamow bag", "Diamox", "अक्सिजन"],
            urgency_level  = 4,
            age_group      = "both",
            language       = "ne",
            source         = "Nepal_MoHP_2078",
            step_type      = "action",
            section        = "लेक लाग्नु — उपचार",
            keywords       = ["लेक", "तल झर्नु", "Dexamethasone", "अक्सिजन", "Gamow"],
        ),
        text=(
            "लेक लाग्नु — उपचार:\\n\\n"
            "सामान्य लेक (AMS):\\n"
            "• उचाइमा जान तुरुन्त रोक्नुस्। उही उचाइमा २४ घण्टा आराम गर्नुस्।\\n"
            "• प्रशस्त पानी पिउनुस् (दिनमा ३–४ लिटर)।\\n"
            "• टाउको दुखाइको लागि सिटामोल वा इबुप्रोफेन।\\n"
            "• Acetazolamide (Diamox) २५० मिलिग्राम दिनमा दुईपटक (उपलब्ध भएमा)।\\n"
            "• पूर्ण निको नभएसम्म माथि कहिल्यै नजानुस्।\\n\\n"
            "गम्भीर लेक / HACE:\\n"
            "• तुरुन्तै तल झर्नुस् — कम्तीमा ५००–१००० मिटर। बिहानसम्म नपर्खनुस्।\\n"
            "• Dexamethasone ८ मिलिग्राम तुरुन्त (उपलब्ध भएमा)।\\n"
            "• अक्सिजन उपलब्ध छ भने दिनुस् (२–४ L/min)।\\n"
            "• Gamow bag (portable hyperbaric chamber) उपलब्ध छ भने प्रयोग गर्नुस्।\\n\\n"
            "HAPE:\\n"
            "• तुरुन्त तल झर्नु — यही एकमात्र सही उपचार हो।\\n"
            "• अक्सिजन ४–६ L/min। Nifedipine (उपलब्ध भएमा)।\\n\\n"
            "सुनौलो नियम: शंका लागे तल झर्नुस् — तल झर्नु नै एकमात्र सही उपचार हो।"
        ),
    ),

    # ══════════════════════════════════════════════════════════════════════
    # 9. DROWNING (डुब्नु)
    # ══════════════════════════════════════════════════════════════════════

    ProtocolChunk(
        metadata=ChunkMetadata(
            chunk_id       = "WHO_BEC_drowning_both_action_01_ne",
            condition      = "drowning",
            condition_tags = ["डुब्नु", "पानीमा डुबेको", "खोलामा बगेको",
                              "पौडी खेल्दा डुब्नु", "पानीबाट निकालेको"],
            urgency_level  = 5,
            age_group      = "both",
            language       = "ne",
            source         = "WHO_BEC_2016",
            step_type      = "action",
            section        = "पानीमा डुब्नु — उद्धार र उपचार",
            keywords       = ["डुब्नु", "सास दिनु", "CPR", "माध्यमिक डुब्नु", "अस्पताल"],
        ),
        text=(
            "पानीमा डुब्नु — उद्धार र प्राथमिक उपचार:\\n"
            "चरण १ — सुरक्षित उद्धार: तालिम नभई गहिरो वा छिटो बग्ने पानीमा नपस्नुस्।\\n"
            "• फाल्नुस्: डोरी, लुगा जोडेर, खाली डिब्बा वा जीवन रक्षा चक्र।\\n"
            "• पुग्नुस्: हात, लौरो, तौलिया वा पेटीले किनारबाट तान्नुस्।\\n"
            "• डुंगा: उपलब्ध छ भने प्रयोग गर्नुस्।\\n\\n"
            "चरण २ — बाहिर निकालेपछि: होस र सास जाँच गर्नुस्।\\n"
            "चरण ३ — सास नफेरेको भए:\\n"
            "• छाती थिच्नुअघि तुरुन्त ५ पटक मुखबाट सास दिनुस् — डुब्नेलाई अक्सिजन पहिले चाहिन्छ।\\n"
            "• त्यसपछि ३०:२ CPR (३० थिचाइ, २ सास) निरन्तर गर्नुस्।\\n"
            "चरण ४ — पेट थिचेर पानी निकाल्ने कोसिस नगर्नुस् — समय खेर जान्छ।\\n"
            "चरण ५ — होस आए पनि अनिवार्य अस्पताल लैजानुस्: "
            "फोक्सोमा पानी गएको छ भने केही घण्टापछि ज्यान जान सक्छ।\\n"
            "चरण ६ — चिसोबाट बचाउनुस्: भिजेको लुगा हटाउनुस्, सुक्खा कपडाले छोप्नुस्।"
        ),
    ),

    # ══════════════════════════════════════════════════════════════════════
    # 10. SHOCK (सक)
    # ══════════════════════════════════════════════════════════════════════

    ProtocolChunk(
        metadata=ChunkMetadata(
            chunk_id       = "WHO_BEC_shock_both_action_01_ne",
            condition      = "shock",
            condition_tags = ["सक", "रक्तचाप घट्नु", "पसिना आउने",
                              "चिसो छाला", "कमजोर नाडी", "ढल्नु"],
            urgency_level  = 5,
            age_group      = "both",
            language       = "ne",
            source         = "WHO_BEC_2016",
            step_type      = "action",
            section        = "सक — पहिचान र उपचार",
            keywords       = ["सक", "सुताउनु", "खुट्टा उठाउनु", "तातो राख्नु"],
        ),
        text=(
            "सक (Shock) — पहिचान र तुरुन्त गर्नुपर्ने काम:\\n\\n"
            "लक्षण:\\n"
            "• छाला पहेंलो र चिसो हुनु, पसिना आउनु।\\n"
            "• नाडी छिटो तर कमजोर चल्नु।\\n"
            "• सास फेर्न छिटो र उथलो हुनु।\\n"
            "• अलमलिनु, बेचैन हुनु वा असामान्य निद्रा आउनु।\\n"
            "• वाकवाकी लाग्नु वा ढल्नु।\\n\\n"
            "उपचार:\\n"
            "चरण १ — बिरामीलाई भुइँमा सिधा सुताउनुस्।\\n"
            "चरण २ — खुट्टाको भाग मुटुभन्दा करिब ३० सेमि माथि उठाउनुस् "
            "(खुट्टा, घाँटी वा मेरुदण्ड भाँचिएको शंका भएमा वा सास फेर्न गाह्रो भएमा नउठाउनुस्)।\\n"
            "चरण ३ — देखिएको रगत बग्ने घाउ दबाएर रोक्नुस्।\\n"
            "चरण ४ — न्यानो कपडाले छोप्नुस्।\\n"
            "चरण ५ — खान वा पिउन केही नदिनुस्।\\n"
            "चरण ६ — एक्लै नछोड्नुस्। तुरुन्त १०२ मा फोन गर्नुस्। "
            "सास फेरे नफेरेको निरन्तर जाँच गर्दै रहनुस्।"
        ),
    ),

    # ══════════════════════════════════════════════════════════════════════
    # 11. SEIZURE (छारे रोग / कम्पन)
    # ══════════════════════════════════════════════════════════════════════

    ProtocolChunk(
        metadata=ChunkMetadata(
            chunk_id       = "WHO_BEC_seizure_both_action_01_ne",
            condition      = "seizure",
            condition_tags = ["छारे रोग", "कम्पन", "काम्ने", "मृगी",
                              "बेहोस भएर काम्नु", "अपस्मार", "फिट"],
            urgency_level  = 4,
            age_group      = "both",
            language       = "ne",
            source         = "WHO_BEC_2016",
            step_type      = "action",
            section        = "छारे रोग / कम्पन — प्राथमिक उपचार",
            keywords       = ["कम्पन", "छारे रोग", "सुरक्षा", "मुखमा नराख्नुस्", "कोल्टे"],
        ),
        text=(
            "छारे रोग (कम्पन) — तुरुन्त गर्नुपर्ने काम:\\n"
            "चरण १ — चोटपटकबाट बचाउनुस्: वरपरका कडा, धारिला वा तातो वस्तु हटाउनुस्। "
            "टाउकोमुनि नरम कपडा वा तकिया राख्नुस्।\\n"
            "चरण २ — कम्पन सुरु भएको समय घडी हेरेर नोट गर्नुस्।\\n"
            "चरण ३ — बिरामीलाई जबर्जस्ती समातेर रोक्ने कोसिस कहिल्यै नगर्नुस्।\\n"
            "चरण ४ — मुखमा केही पनि नराख्नुस् — चम्चा, औंला वा कपडा पनि होइन। "
            "यसले दाँत भाँचिन्छ र श्वासनली बन्द हुन सक्छ। "
            "मान्छेले जिब्रो निल्दैन।\\n"
            "चरण ५ — कम्पन रोकिएपछि: कोल्टे फेरेर सुताउनुस् (Recovery Position) — "
            "यसले र्याल वा बान्ताले श्वासनली बन्द हुनबाट जोगाउँछ।\\n"
            "चरण ६ — बसेर जाँच गर्दै रहनुस्। बिरामी अलमलिन सक्छन् — यो सामान्य हो।\\n\\n"
            "तुरुन्त १०२ मा फोन गर्नुस् यदि:\\n"
            "• कम्पन ५ मिनेटभन्दा बढी समय भयो।\\n"
            "• एकपटक रोकिएर फेरि सुरु भयो।\\n"
            "• कम्पन रोकिएपछि पनि होस आएन।\\n"
            "• कम्पनको समयमा चोट लाग्यो।\\n"
            "• गर्भवती महिला हुन् वा पानीमा काँपेको हो।\\n"
            "• पहिलोपटक कम्पन आएको हो।"
        ),
    ),

    ProtocolChunk(
        metadata=ChunkMetadata(
            chunk_id       = "WHO_BEC_seizure_both_warning_01_ne",
            condition      = "seizure",
            condition_tags = ["छारे रोग चेतावनी", "समाउनु हुँदैन",
                              "मुखमा राख्नु हुँदैन", "जिब्रो निल्दैन"],
            urgency_level  = 4,
            age_group      = "both",
            language       = "ne",
            source         = "WHO_BEC_2016",
            step_type      = "warning",
            section        = "छारे रोग — गर्न नहुने काम",
            keywords       = ["कम्पन चेतावनी", "चम्चा", "समाउनु", "जिब्रो"],
        ),
        text=(
            "छारे रोग (कम्पन) — गर्न नहुने काम:\\n"
            "बिरामीलाई जबर्जस्ती समाउनु वा रोक्नु हुँदैन।\\n"
            "मुखमा चम्चा, औंला, कपडा वा कुनै पनि वस्तु राख्नु हुँदैन — "
            "यसले दाँत भाँचिन्छ, औंला टोकिन्छ र श्वासनली बन्द हुन सक्छ।\\n"
            "कम्पन हुँदा जिब्रो निल्दैन — यो गलत धारणा हो।\\n"
            "पानी वा औषधि दिनु हुँदैन (कम्पन हुँदा वा तुरुन्त पछि)।\\n"
            "एक्लै छोडेर जानु हुँदैन।\\n"
            "थप्पड हान्नु वा मुखमा पानी छ्याप्नु हुँदैन।"
        ),
    ),

]
'''

with open("/kaggle/working/neva/rag/seed_protocols_ne.py", "w") as f:
    f.write(content)

# ── Verify ─────────────────────────────────────────────────────────────────────

import importlib
import sys
sys.path.insert(0, "/kaggle/working/neva")

import rag.seed_protocols_ne
importlib.reload(rag.seed_protocols_ne)
from rag.seed_protocols_ne import SEED_CHUNKS_NE

print(f"✓ seed_protocols_ne.py written and loaded")
print(f"✓ Total Nepali chunks: {len(SEED_CHUNKS_NE)}")

from collections import Counter
condition_counts = Counter(c.metadata.condition for c in SEED_CHUNKS_NE)
step_type_counts = Counter(c.metadata.step_type  for c in SEED_CHUNKS_NE)

print(f"\nNepali chunks by condition:")
for cond, count in sorted(condition_counts.items()):
    print(f"  {cond:30s} {count}")

print(f"\nNepali chunks by step_type:")
for stype, count in sorted(step_type_counts.items()):
    print(f"  {stype:20s} {count}")

# Verify all conditions covered
from rag.seed_protocols import SEED_CHUNKS
en_conditions = {c.metadata.condition for c in SEED_CHUNKS}
ne_conditions  = {c.metadata.condition for c in SEED_CHUNKS_NE}
missing = en_conditions - ne_conditions

if missing:
    print(f"\n✗ Missing Nepali chunks for: {missing}")
else:
    print(f"\n✓ All {len(en_conditions)} conditions covered in both English and Nepali")

✓ seed_protocols_ne.py written and loaded
✓ Total Nepali chunks: 20

Nepali chunks by condition:
  altitude_sickness              2
  burns                          2
  chest_pain                     2
  choking                        2
  drowning                       1
  seizure                        2
  severe_bleeding                2
  shock                          1
  snakebite                      2
  stroke                         2
  unconscious                    2

Nepali chunks by step_type:
  action               12
  assessment           3
  warning              5

✓ All 11 conditions covered in both English and Nepali


In [7]:
# CELL 6 — Patch to_chroma_meta() to strip None values, then rebuild

import sys, json
sys.path.insert(0, "/kaggle/working/neva")

# ── Step 1: Patch models.py with a None-safe to_chroma_meta() ────────────────

models_content = '''"""
NEVA RAG — Pydantic models.
Contract between: chunker → ChromaDB → retriever → Gemma prompt builder.
"""

from __future__ import annotations
from typing import Literal, Optional
from pydantic import BaseModel, Field


# ── Type aliases ──────────────────────────────────────────────────────────────

StepType     = Literal["overview", "assessment", "action", "warning", "do_not"]
AgeGroup     = Literal["adult", "paediatric", "both"]
LanguageCode = Literal["en", "ne"]
Source       = Literal[
    "WHO_BEC_2016",
    "Nepal_MoHP_2078",
    "WHO_PHEC_2026",
    "NEVA_SEED",
]


# ── Chunk metadata ────────────────────────────────────────────────────────────

class ChunkMetadata(BaseModel):
    chunk_id:       str
    condition:      str
    condition_tags: list[str]       = Field(default_factory=list)
    urgency_level:  int             = Field(..., ge=1, le=5)
    age_group:      AgeGroup
    language:       LanguageCode
    source:         Source
    step_type:      StepType
    section:        str
    page_ref:       Optional[str]   = None
    keywords:       list[str]       = Field(default_factory=list)

    def to_chroma_meta(self) -> dict:
        """
        ChromaDB Rust backend only accepts: str, int, float, bool.
        Rules applied here:
          - list[str]    → pipe-joined str
          - None         → empty string ""
          - everything else passes through as-is
        """
        d = self.model_dump()

        # Flatten lists
        d["condition_tags"] = "|".join(d["condition_tags"]) if d["condition_tags"] else ""
        d["keywords"]       = "|".join(d["keywords"])       if d["keywords"]       else ""

        # Replace ANY remaining None with ""
        sanitised = {}
        for k, v in d.items():
            if v is None:
                sanitised[k] = ""
            elif isinstance(v, list):
                # Safety net: should not reach here, but flatten anyway
                sanitised[k] = "|".join(str(x) for x in v)
            else:
                sanitised[k] = v

        return sanitised


# ── Protocol chunk ────────────────────────────────────────────────────────────

class ProtocolChunk(BaseModel):
    metadata: ChunkMetadata
    text:     str


# ── Retrieval request ─────────────────────────────────────────────────────────

class RetrievalRequest(BaseModel):
    query:         str
    condition:     Optional[str]      = None
    urgency_level: Optional[int]      = Field(None, ge=1, le=5)
    age_group:     Optional[AgeGroup] = None
    language:      LanguageCode       = "en"
    top_k:         int                = Field(5, ge=1, le=15)


# ── Retrieval result ──────────────────────────────────────────────────────────

class RetrievedChunk(BaseModel):
    chunk_id:      str
    condition:     str
    urgency_level: int
    step_type:     StepType
    source:        str
    text:          str
    score:         float

class RetrievalResult(BaseModel):
    query:    str
    chunks:   list[RetrievedChunk]
    grounded: bool
'''

with open("/kaggle/working/neva/rag/models.py", "w") as f:
    f.write(models_content)
print("✓ models.py patched")

# ── Step 2: Verify the fix works on a sample chunk ───────────────────────────

# Force reload so the patched version is used
import importlib
import rag.models
importlib.reload(rag.models)

# Also reload seed files so they use the new ChunkMetadata
import rag.seed_protocols
import rag.seed_protocols_ne
importlib.reload(rag.seed_protocols)
importlib.reload(rag.seed_protocols_ne)

from rag.models            import ChunkMetadata, ProtocolChunk
from rag.seed_protocols    import SEED_CHUNKS
from rag.seed_protocols_ne import SEED_CHUNKS_NE

print("\nVerifying to_chroma_meta() output on every chunk...")
all_chunks = SEED_CHUNKS + SEED_CHUNKS_NE
issues = []

for chunk in all_chunks:
    flat = chunk.metadata.to_chroma_meta()
    for k, v in flat.items():
        if v is None:
            issues.append(f"  None value found: chunk={chunk.metadata.chunk_id} key={k}")
        if isinstance(v, list):
            issues.append(f"  List value found: chunk={chunk.metadata.chunk_id} key={k} val={v}")
        if not isinstance(v, (str, int, float, bool)):
            issues.append(f"  Bad type: chunk={chunk.metadata.chunk_id} key={k} type={type(v)}")

if issues:
    print("✗ Issues found:")
    for i in issues:
        print(i)
else:
    print(f"✓ All {len(all_chunks)} chunks produce valid flat metadata")

# Print one sample to confirm
sample = all_chunks[0].metadata.to_chroma_meta()
print(f"\nSample flat metadata for '{all_chunks[0].metadata.chunk_id}':")
for k, v in sample.items():
    print(f"  {k:20s} : {repr(v)}")

# ── Step 3: Clear all caches ──────────────────────────────────────────────────

try:
    import rag.retriever as _ret
    importlib.reload(_ret)
    _ret._get_collection.cache_clear()
    _ret._get_model.cache_clear()
    print("\n✓ Retriever cache cleared")
except Exception as e:
    print(f"\n(Retriever not yet loaded: {e})")

# ── Step 4: Rebuild ChromaDB from scratch ─────────────────────────────────────

from pathlib import Path
import chromadb
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer

CHROMA_PATH     = "/kaggle/working/neva/chroma_db"
COLLECTION_NAME = "neva_protocols"
EMBED_MODEL     = "BAAI/bge-m3"
QUERY_PREFIX    = "Represent this sentence for searching relevant passages: "

print("\nRebuilding ChromaDB...")
client = chromadb.PersistentClient(
    path     = CHROMA_PATH,
    settings = Settings(anonymized_telemetry=False),
)

# Clean slate
existing = [c.name for c in client.list_collections()]
if COLLECTION_NAME in existing:
    client.delete_collection(COLLECTION_NAME)
    print(f"  ✓ Deleted stale collection")

collection = client.create_collection(
    name     = COLLECTION_NAME,
    metadata = {"hnsw:space": "cosine"},
)
print(f"  ✓ Created fresh collection")

# Embed
print(f"\nEmbedding {len(all_chunks)} chunks with {EMBED_MODEL}...")
model = SentenceTransformer(EMBED_MODEL)
texts = [c.text for c in all_chunks]

embeddings = model.encode(
    texts,
    normalize_embeddings = True,
    show_progress_bar    = True,
    batch_size           = 16,
).tolist()
print(f"  ✓ {len(embeddings)} vectors, dim={len(embeddings[0])}")

# Insert — with per-item validation so any bad chunk is immediately identified
print(f"\nInserting chunks...")
failed_chunks = []

for i, (chunk, emb) in enumerate(zip(all_chunks, embeddings)):
    flat_meta = chunk.metadata.to_chroma_meta()

    # Final type check before sending to Rust
    bad_fields = {k: v for k, v in flat_meta.items()
                  if not isinstance(v, (str, int, float, bool))}
    if bad_fields:
        print(f"  ✗ Skipping {chunk.metadata.chunk_id} — bad fields: {bad_fields}")
        failed_chunks.append(chunk.metadata.chunk_id)
        continue

    try:
        collection.add(
            ids        = [chunk.metadata.chunk_id],
            embeddings = [emb],
            documents  = [chunk.text],
            metadatas  = [flat_meta],
        )
    except Exception as e:
        print(f"  ✗ Failed on {chunk.metadata.chunk_id}: {e}")
        failed_chunks.append(chunk.metadata.chunk_id)

# ── Step 5: Verify ────────────────────────────────────────────────────────────
final_count = collection.count()
expected    = len(all_chunks) - len(failed_chunks)

print(f"\n{'='*60}")
print(f"  Attempted : {len(all_chunks)}")
print(f"  Failed    : {len(failed_chunks)}")
print(f"  Inserted  : {final_count}")

if len(failed_chunks) > 0:
    print(f"\n  ✗ Failed chunk IDs:")
    for cid in failed_chunks:
        print(f"    - {cid}")
else:
    print(f"\n  ✓ All chunks inserted successfully")

assert final_count == expected, f"Count mismatch: got {final_count}, expected {expected}"
print(f"  ✓ Count verified")

# ── Step 6: Quick retrieval smoke test ───────────────────────────────────────
print(f"\nSmoke test: querying collection directly...")

test_vec = model.encode(
    QUERY_PREFIX + "person is choking cannot breathe",
    normalize_embeddings = True,
).tolist()

results = collection.query(
    query_embeddings = [test_vec],
    n_results        = 3,
    include          = ["documents", "metadatas", "distances"],
)

print(f"  Top 3 results for 'person is choking cannot breathe':")
for cid, meta, dist in zip(
    results["ids"][0],
    results["metadatas"][0],
    results["distances"][0],
):
    print(f"  [{dist:.4f}] {cid}  ({meta['condition']}, {meta['step_type']})")

print(f"\n{'='*60}")
print(f"  ✓ FIX COMPLETE — safe to run Cell 7 now")
print(f"{'='*60}")

✓ models.py patched

Verifying to_chroma_meta() output on every chunk...
✓ All 42 chunks produce valid flat metadata

Sample flat metadata for 'WHO_BEC_choking_adult_assessment_01':
  chunk_id             : 'WHO_BEC_choking_adult_assessment_01'
  condition            : 'choking'
  condition_tags       : 'airway obstruction|foreign body|unable to breathe|gagging|silent cough|turning blue|cannot speak'
  urgency_level        : 5
  age_group            : 'adult'
  language             : 'en'
  source               : 'WHO_BEC_2016'
  step_type            : 'assessment'
  section              : 'Choking — Recognition'
  page_ref             : ''
  keywords             : 'choking|airway|obstruction|cough|silent|speak'

(Retriever not yet loaded: No module named 'rag.retriever')

Rebuilding ChromaDB...
  ✓ Created fresh collection

Embedding 42 chunks with BAAI/bge-m3...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

  ✓ 42 vectors, dim=1024

Inserting chunks...

  Attempted : 42
  Failed    : 0
  Inserted  : 42

  ✓ All chunks inserted successfully
  ✓ Count verified

Smoke test: querying collection directly...
  Top 3 results for 'person is choking cannot breathe':
  [0.3931] WHO_BEC_choking_adult_assessment_01  (choking, assessment)
  [0.4433] WHO_BEC_choking_adult_action_01  (choking, action)
  [0.4900] WHO_BEC_unconscious_adult_action_01  (unconscious, action)

  ✓ FIX COMPLETE — safe to run Cell 7 now


In [8]:
# Cell 7 — Write rag/retriever.py

content = '''"""
NEVA RAG — Core retrieval module.
This is the ONLY module the rest of NEVA imports from the RAG layer.
"""

from __future__ import annotations
from functools import lru_cache
from pathlib import Path

import chromadb
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer

from rag.models import RetrievalRequest, RetrievalResult, RetrievedChunk

# ── Config ────────────────────────────────────────────────────────────────────

CHROMA_PATH     = "/kaggle/working/neva/chroma_db"
COLLECTION_NAME = "neva_protocols"
EMBED_MODEL     = "BAAI/bge-m3"
QUERY_PREFIX    = "Represent this sentence for searching relevant passages: "
SCORE_THRESHOLD = 0.75   # cosine distance (lower = better match)


# ── Lazy singletons ───────────────────────────────────────────────────────────

@lru_cache(maxsize=1)
def _get_model() -> SentenceTransformer:
    print(f"[retriever] Loading {EMBED_MODEL} ...")
    return SentenceTransformer(EMBED_MODEL)


@lru_cache(maxsize=1)
def _get_collection():
    client = chromadb.PersistentClient(
        path     = CHROMA_PATH,
        settings = Settings(anonymized_telemetry=False),
    )
    return client.get_collection(COLLECTION_NAME)


# ── Main retrieval function ───────────────────────────────────────────────────

def retrieve(request: RetrievalRequest) -> RetrievalResult:
    """
    Core retrieval function called by the Gemma reasoning stage.

    1. Embeds the query with bge-m3.
    2. Applies metadata filters (language, age_group, urgency_level).
    3. Returns top-k chunks ranked by cosine similarity.
    4. Sets grounded=True if at least one chunk is below SCORE_THRESHOLD.
    """
    model      = _get_model()
    collection = _get_collection()

    # ── Embed query ───────────────────────────────────────────────────────────
    query_vec = model.encode(
        QUERY_PREFIX + request.query,
        normalize_embeddings = True,
    ).tolist()

    # ── Build metadata filter ─────────────────────────────────────────────────
    filters = []

    if request.language:
        filters.append({"language": {"$eq": request.language}})

    if request.age_group and request.age_group != "both":
        filters.append({"age_group": {"$in": [request.age_group, "both"]}})

    if request.urgency_level is not None:
        filters.append({"urgency_level": {"$gte": request.urgency_level}})

    where = (
        {"$and": filters} if len(filters) > 1
        else filters[0]   if len(filters) == 1
        else None
    )

    # ── Query ChromaDB ────────────────────────────────────────────────────────
    def _query(where_clause):
        kwargs = dict(
            query_embeddings = [query_vec],
            n_results        = request.top_k,
            include          = ["documents", "metadatas", "distances"],
        )
        if where_clause:
            kwargs["where"] = where_clause
        return collection.query(**kwargs)

    try:
        results = _query(where)
    except Exception as e:
        # Filter may yield 0 results in ChromaDB — fall back to unfiltered
        print(f"[retriever] Filter failed ({e}), falling back to unfiltered query")
        results = _query(None)

    # ── Parse results ─────────────────────────────────────────────────────────
    docs      = results["documents"][0]
    metas     = results["metadatas"][0]
    distances = results["distances"][0]

    chunks = [
        RetrievedChunk(
            chunk_id      = m["chunk_id"],
            condition     = m["condition"],
            urgency_level = int(m["urgency_level"]),
            step_type     = m["step_type"],
            source        = m["source"],
            text          = doc,
            score         = round(dist, 4),
        )
        for doc, m, dist in zip(docs, metas, distances)
    ]

    grounded = any(c.score < SCORE_THRESHOLD for c in chunks)

    return RetrievalResult(
        query    = request.query,
        chunks   = chunks,
        grounded = grounded,
    )


# ── Prompt formatter ──────────────────────────────────────────────────────────

def format_for_prompt(result: RetrievalResult, max_chunks: int = 4) -> str:
    """
    Formats retrieved chunks into the block injected into Gemma\\'s system prompt.
    Only includes chunks that passed the score threshold (verified retrieval).
    If no chunk passes: returns a hard refusal block.
    """
    verified = [c for c in result.chunks if c.score < SCORE_THRESHOLD][:max_chunks]

    if not verified:
        return (
            "[PROTOCOL RETRIEVAL FAILED: No verified protocol matched this query.\\n"
            "DO NOT provide medical advice.\\n"
            "Tell the user: I do not have a verified protocol for this situation. "
            "Please call emergency services immediately — dial 102.]"
        )

    lines = [
        "══════════════════════════════════════════════════════",
        "VERIFIED MEDICAL PROTOCOLS — Use ONLY the information",
        "below. Do not add, invent, or infer beyond this.     ",
        "══════════════════════════════════════════════════════",
        "",
    ]

    for i, chunk in enumerate(verified, 1):
        lines += [
            f"--- Protocol {i} of {len(verified)} ---",
            f"Condition     : {chunk.condition}",
            f"Urgency       : {chunk.urgency_level}/5",
            f"Type          : {chunk.step_type}",
            f"Source        : {chunk.source}",
            f"Match score   : {chunk.score:.4f} (lower = better)",
            "",
            chunk.text,
            "",
        ]

    lines += [
        "══════════════════════════════════════════════════════",
        "END OF VERIFIED PROTOCOLS",
        "══════════════════════════════════════════════════════",
    ]

    return "\\n".join(lines)
'''

with open("/kaggle/working/neva/rag/retriever.py", "w") as f:
    f.write(content)

# Quick smoke test
import importlib
import rag.retriever
importlib.reload(rag.retriever)

from rag.retriever import retrieve, format_for_prompt
from rag.models    import RetrievalRequest

test_req = RetrievalRequest(
    query         = "person is choking cannot breathe",
    condition     = "choking",
    urgency_level = 5,
    age_group     = "adult",
    language      = "en",
    top_k         = 3,
)

result = retrieve(test_req)
print(f"✓ retriever.py smoke test passed")
print(f"  Grounded      : {result.grounded}")
print(f"  Chunks returned: {len(result.chunks)}")
for c in result.chunks:
    print(f"  [{c.score:.4f}] {c.chunk_id}")

[retriever] Loading BAAI/bge-m3 ...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

✓ retriever.py smoke test passed
  Grounded      : True
  Chunks returned: 3
  [0.3931] WHO_BEC_choking_adult_assessment_01
  [0.4433] WHO_BEC_choking_adult_action_01
  [0.4900] WHO_BEC_unconscious_adult_action_01


In [9]:
# Cell 8 — Write rag/rag_endpoint.py

content = '''"""
NEVA RAG — FastAPI endpoint.
Mounts at /rag on the main NEVA API, or runs standalone for testing.
"""

import sys
sys.path.insert(0, "/kaggle/working/neva")

from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware

from rag.models    import RetrievalRequest, RetrievalResult
from rag.retriever import retrieve, format_for_prompt

app = FastAPI(
    title       = "NEVA RAG API",
    description = "Retrieval endpoint for Nepal Emergency Voice Assistant",
    version     = "1.0.0",
)

app.add_middleware(
    CORSMiddleware,
    allow_origins  = ["*"],
    allow_methods  = ["*"],
    allow_headers  = ["*"],
)


@app.get("/health")
def health():
    return {"status": "ok", "service": "NEVA-RAG"}


@app.post("/rag/retrieve", response_model=RetrievalResult)
def retrieve_endpoint(request: RetrievalRequest) -> RetrievalResult:
    """
    Primary retrieval endpoint.
    Returns ranked protocol chunks for the given emergency query.
    Called by the Gemma reasoning stage before prompt construction.
    """
    try:
        return retrieve(request)
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))


@app.post("/rag/prompt-block")
def prompt_block_endpoint(request: RetrievalRequest) -> dict:
    """
    Convenience endpoint.
    Returns a pre-formatted string ready to inject into the Gemma system prompt.
    """
    try:
        result = retrieve(request)
        block  = format_for_prompt(result)
        return {
            "prompt_block"  : block,
            "grounded"      : result.grounded,
            "verified_chunks": len([c for c in result.chunks if c.score < 0.75]),
        }
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))
'''

with open("/kaggle/working/neva/rag/rag_endpoint.py", "w") as f:
    f.write(content)

print("✓ rag_endpoint.py written")
print("\nStarting FastAPI server for testing...")

# Start server in background thread for Kaggle testing
import threading
import time
import nest_asyncio
import uvicorn

nest_asyncio.apply()

def run_server():
    sys.path.insert(0, "/kaggle/working/neva")
    from rag.rag_endpoint import app
    uvicorn.run(app, host="0.0.0.0", port=8001, log_level="warning")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(3)

# Quick API health check
import requests
try:
    r = requests.get("http://localhost:8001/health")
    print(f"✓ API health check: {r.json()}")
except Exception as e:
    print(f"✗ API not reachable: {e}")

✓ rag_endpoint.py written

Starting FastAPI server for testing...
✓ API health check: {'status': 'ok', 'service': 'NEVA-RAG'}


In [10]:
# Cell 9 — Full retrieval test suite (8 conditions)

import sys
sys.path.insert(0, "/kaggle/working/neva")

from rag.models    import RetrievalRequest
from rag.retriever import retrieve, format_for_prompt

TEST_CASES = [
    {
        "name":              "1. Severe Bleeding",
        "request": RetrievalRequest(
            query         = "There is a lot of blood, the person has a deep cut on their leg and blood is not stopping",
            condition     = "severe_bleeding",
            urgency_level = 5,
            age_group     = "both",
            language      = "en",
            top_k         = 4,
        ),
        "expect_condition":  "severe_bleeding",
        "expect_step_types": {"action", "warning"},
    },
    {
        "name":              "2. Choking Adult",
        "request": RetrievalRequest(
            query         = "Person is choking, cannot breathe, cannot speak, turning blue, grabbing throat",
            condition     = "choking",
            urgency_level = 5,
            age_group     = "adult",
            language      = "en",
            top_k         = 4,
        ),
        "expect_condition":  "choking",
        "expect_step_types": {"action"},
    },
    {
        "name":              "3. Unconscious Not Breathing",
        "request": RetrievalRequest(
            query         = "Man collapsed on floor, not responding at all, does not appear to be breathing",
            urgency_level = 5,
            age_group     = "adult",
            language      = "en",
            top_k         = 4,
        ),
        "expect_condition":  "unconscious",
        "expect_step_types": {"action"},
    },
    {
        "name":              "4. Snakebite",
        "request": RetrievalRequest(
            query         = "Snake has bitten my friend on the ankle in the field, what do I do right now",
            condition     = "snakebite",
            urgency_level = 5,
            language      = "en",
            top_k         = 4,
        ),
        "expect_condition":  "snakebite",
        "expect_step_types": {"action", "warning"},
    },
    {
        "name":              "5. Chest Pain Heart Attack",
        "request": RetrievalRequest(
            query         = "Elderly man has crushing chest pain going to left arm, sweating heavily, nausea",
            condition     = "chest_pain",
            urgency_level = 5,
            age_group     = "adult",
            language      = "en",
            top_k         = 3,
        ),
        "expect_condition":  "chest_pain",
        "expect_step_types": {"action"},
    },
    {
        "name":              "6. Burns (Child)",
        "request": RetrievalRequest(
            query         = "Child spilled boiling water on arm, large red blistering area, child is crying",
            condition     = "burns",
            urgency_level = 4,
            age_group     = "paediatric",
            language      = "en",
            top_k         = 4,
        ),
        "expect_condition":  "burns",
        "expect_step_types": {"action", "warning"},
    },
    {
        "name":              "7. Altitude Sickness HACE",
        "request": RetrievalRequest(
            query         = "Trekker at high altitude, severe headache, very confused, cannot walk in straight line",
            condition     = "altitude_sickness",
            urgency_level = 4,
            language      = "en",
            top_k         = 3,
        ),
        "expect_condition":  "altitude_sickness",
        "expect_step_types": {"action"},
    },
    {
        "name":              "8. Drowning",
        "request": RetrievalRequest(
            query         = "Child pulled from river, completely unconscious, not breathing after drowning",
            condition     = "drowning",
            urgency_level = 5,
            age_group     = "paediatric",
            language      = "en",
            top_k         = 4,
        ),
        "expect_condition":  "drowning",
        "expect_step_types": {"action"},
    },
    {
        "name":              "9. Snakebite in Nepali",
        "request": RetrievalRequest(
            query         = "साँपले टोक्यो, धेरै दुखेको छ, के गर्ने",
            condition     = "snakebite",
            urgency_level = 5,
            language      = "ne",
            top_k         = 3,
        ),
        "expect_condition":  "snakebite",
        "expect_step_types": {"action"},
    },
    {
        "name":              "10. Choking in Nepali",
        "request": RetrievalRequest(
            query         = "मान्छे दम थिचिएको छ, सास लिन सक्दैन, नीलो हुँदैछ",
            condition     = "choking",
            urgency_level = 5,
            language      = "ne",
            top_k         = 3,
        ),
        "expect_condition":  "choking",
        "expect_step_types": {"action"},
    },
]


# ── Run tests ─────────────────────────────────────────────────────────────────

def run_test(case: dict, show_prompt_preview: bool = False) -> bool:
    print(f"\n{'─'*65}")
    print(f"  {case['name']}")
    print(f"  Query: {case['request'].query[:70]}...")

    result = retrieve(case["request"])
    passed = True

    # Check grounding
    if not result.grounded:
        print(f"  ✗ FAIL: No chunk passed score threshold (all scores ≥ 0.75)")
        passed = False

    # Check top condition
    top_condition = result.chunks[0].condition if result.chunks else None
    if "expect_condition" in case and top_condition != case["expect_condition"]:
        print(f"  ✗ FAIL: Expected '{case['expect_condition']}', got '{top_condition}'")
        passed = False

    # Check step_types in verified results
    found_types = {c.step_type for c in result.chunks if c.score < 0.75}
    for expected in case.get("expect_step_types", set()):
        if expected not in found_types:
            print(f"  ✗ FAIL: step_type '{expected}' not in verified results")
            passed = False

    # Print chunk summary
    status_icon = "✓ PASS" if passed else "✗ FAIL"
    print(f"  {status_icon}  |  grounded={result.grounded}  |  {len(result.chunks)} chunks returned")
    for chunk in result.chunks:
        verified_mark = "●" if chunk.score < 0.75 else "○"
        print(f"    {verified_mark} [{chunk.score:.4f}]  {chunk.chunk_id}")

    if show_prompt_preview and passed:
        print(f"\n  --- Prompt block preview (first verified chunk only) ---")
        preview = format_for_prompt(result, max_chunks=1)
        print(f"  {preview[:300]}...")

    return passed


# Run all
print("="*65)
print("  NEVA RAG — Full Retrieval Test Suite")
print("="*65)

results_list = [run_test(c, show_prompt_preview=(i == 0)) for i, c in enumerate(TEST_CASES)]

total  = len(results_list)
passed = sum(results_list)
failed = total - passed

print(f"\n{'='*65}")
print(f"  RESULTS: {passed}/{total} passed  |  {failed} failed")
if failed == 0:
    print("  ✓ All tests passed. RAG layer is ready for integration.")
else:
    failed_cases = [TEST_CASES[i]['name'] for i, r in enumerate(results_list) if not r]
    print(f"  ✗ Failed cases: {failed_cases}")
print(f"{'='*65}")

  NEVA RAG — Full Retrieval Test Suite

─────────────────────────────────────────────────────────────────
  1. Severe Bleeding
  Query: There is a lot of blood, the person has a deep cut on their leg and bl...
  ✓ PASS  |  grounded=True  |  4 chunks returned
    ● [0.4584]  WHO_BEC_bleeding_both_action_01
    ● [0.5281]  WHO_BEC_bleeding_both_warning_01
    ● [0.5468]  WHO_BEC_choking_adult_assessment_01
    ● [0.5590]  WHO_BEC_shock_both_action_01

  --- Prompt block preview (first verified chunk only) ---
  ══════════════════════════════════════════════════════
VERIFIED MEDICAL PROTOCOLS — Use ONLY the information
below. Do not add, invent, or infer beyond this.     
══════════════════════════════════════════════════════

--- Protocol 1 of 1 ---
Condition     : severe_bleeding
Urgency       : 5/5
Type ...

─────────────────────────────────────────────────────────────────
  2. Choking Adult
  Query: Person is choking, cannot breathe, cannot speak, turning blue, grabbin...
  ✓ PASS  | 

In [11]:
# Cell 10 — End-to-end integration demo
# Shows exactly what Gemma will receive from the RAG layer

import sys, json
sys.path.insert(0, "/kaggle/working/neva")

from rag.models    import RetrievalRequest
from rag.retriever import retrieve, format_for_prompt

print("="*65)
print("NEVA RAG — End-to-End Integration Demo")
print("Simulating: Stage 1 (extraction) → RAG → Gemma prompt block")
print("="*65)

# ── Simulate Stage 1 extraction output ────────────────────────────────────────
# In production this comes from Gemma's extraction stage.
# Here we hard-code it to demo the RAG output.

extraction_outputs = [
    {
        "label":       "Snakebite (English)",
        "condition":     "snakebite",
        "urgency_level": 5,
        "age_group":     "adult",
        "language":      "en",
        "raw_query":     "A snake just bit my friend in the field, his leg is swelling fast",
    },
    {
        "label":       "साँप टोकेको (Nepali)",
        "condition":     "snakebite",
        "urgency_level": 5,
        "age_group":     "both",
        "language":      "ne",
        "raw_query":     "साँपले टोक्यो, खुट्टा सुन्निँदैछ, के गर्ने",
    },
    {
        "label":       "Unconscious child (English)",
        "condition":     "unconscious",
        "urgency_level": 5,
        "age_group":     "paediatric",
        "language":      "en",
        "raw_query":     "My child fell and is not waking up, not breathing",
    },
]

for case in extraction_outputs:
    print(f"\n{'─'*65}")
    print(f"SCENARIO: {case['label']}")
    print(f"Query   : {case['raw_query']}")

    result = retrieve(RetrievalRequest(
        query         = case["raw_query"],
        condition     = case["condition"],
        urgency_level = case["urgency_level"],
        age_group     = case.get("age_group"),
        language      = case["language"],
        top_k         = 4,
    ))

    prompt_block = format_for_prompt(result, max_chunks=3)

    print(f"\n--- WHAT GEMMA RECEIVES (system prompt injection) ---\n")
    print(prompt_block)

    # Show the full system prompt template
    SYSTEM_PROMPT_TEMPLATE = f"""You are NEVA, the Nepal Emergency Voice Assistant.
You give calm, clear, step-by-step first-aid guidance to panicked bystanders.

STRICT RULES:
1. Use ONLY the verified protocols provided below. Never invent or add medical information.
2. If the protocols do not cover the situation, say exactly:
   "I don't have a verified protocol for this. Call 102 immediately."
3. Speak simply — one instruction per sentence. No jargon.
4. Always end with: "Call 102 now if you haven't already."
5. Language: respond in {"Nepali" if case["language"] == "ne" else "English"}.

{prompt_block}"""

    print(f"\n--- FULL SYSTEM PROMPT THAT GEMMA RECEIVES ---\n")
    print(SYSTEM_PROMPT_TEMPLATE[:800] + "\n...[truncated for display]")

print(f"\n{'='*65}")
print("✓ RAG integration demo complete.")
print("✓ These prompt blocks are ready to be passed to Gemma.")
print(f"{'='*65}")

NEVA RAG — End-to-End Integration Demo
Simulating: Stage 1 (extraction) → RAG → Gemma prompt block

─────────────────────────────────────────────────────────────────
SCENARIO: Snakebite (English)
Query   : A snake just bit my friend in the field, his leg is swelling fast

--- WHAT GEMMA RECEIVES (system prompt injection) ---

══════════════════════════════════════════════════════
VERIFIED MEDICAL PROTOCOLS — Use ONLY the information
below. Do not add, invent, or infer beyond this.     
══════════════════════════════════════════════════════

--- Protocol 1 of 3 ---
Condition     : snakebite
Urgency       : 5/5
Type          : action
Source        : Nepal_MoHP_2078
Match score   : 0.4488 (lower = better)

SNAKEBITE — IMMEDIATE FIRST AID (Nepal MoHP):
Step 1 — Move person away from snake. Do NOT attempt to catch or kill the snake.
Step 2 — Keep person CALM and as STILL as possible. Movement increases blood flow and spreads venom faster.
Step 3 — Immobilise the bitten limb as if it were a 

In [12]:
# missing build_db.py file

content = '''"""
NEVA RAG — Build (or rebuild) the ChromaDB vector store.
Loads seed chunks (EN + NE) + any JSON files in data/chunks/.
Embedding model: BAAI/bge-m3
"""

import json
import sys
from pathlib import Path

sys.path.insert(0, "/kaggle/working/neva")

import chromadb
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer

from rag.seed_protocols    import SEED_CHUNKS
from rag.seed_protocols_ne import SEED_CHUNKS_NE
from rag.models import ProtocolChunk, ChunkMetadata

CHROMA_PATH     = "/kaggle/working/neva/chroma_db"
COLLECTION_NAME = "neva_protocols"
EMBED_MODEL     = "BAAI/bge-m3"
QUERY_PREFIX    = "Represent this sentence for searching relevant passages: "
CHUNK_DIR       = Path("/kaggle/working/neva/data/chunks")
BATCH_SIZE      = 32


def build(force_rebuild: bool = False):
    client = chromadb.PersistentClient(
        path     = CHROMA_PATH,
        settings = Settings(anonymized_telemetry=False),
    )

    if force_rebuild:
        try:
            client.delete_collection(COLLECTION_NAME)
            print(f"[build_db] Deleted existing collection")
        except Exception:
            pass

    collection = client.get_or_create_collection(
        name     = COLLECTION_NAME,
        metadata = {"hnsw:space": "cosine"},
    )

    all_chunks: list[ProtocolChunk] = []
    all_chunks.extend(SEED_CHUNKS)
    all_chunks.extend(SEED_CHUNKS_NE)

    if CHUNK_DIR.exists():
        for json_file in CHUNK_DIR.glob("*.json"):
            with open(json_file, encoding="utf-8") as f:
                raw = json.load(f)
            for item in raw:
                try:
                    meta  = ChunkMetadata(**{k: v for k, v in item.items() if k != "text"})
                    chunk = ProtocolChunk(metadata=meta, text=item["text"])
                    all_chunks.append(chunk)
                except Exception as e:
                    print(f"[build_db] Skipping malformed chunk: {e}")

    existing_ids = set(collection.get(include=[])["ids"])
    new_chunks   = [c for c in all_chunks if c.metadata.chunk_id not in existing_ids]

    print(f"[build_db] Total chunks : {len(all_chunks)}")
    print(f"[build_db] New chunks   : {len(new_chunks)}")

    if not new_chunks:
        print(f"[build_db] Nothing new. Collection has {collection.count()} chunks.")
        return collection

    model  = SentenceTransformer(EMBED_MODEL)
    texts  = [c.text for c in new_chunks]
    embeddings = model.encode(
        texts,
        normalize_embeddings = True,
        show_progress_bar    = True,
        batch_size           = 16,
    ).tolist()

    for i, (chunk, emb) in enumerate(zip(new_chunks, embeddings)):
        flat = chunk.metadata.to_chroma_meta()
        collection.add(
            ids        = [chunk.metadata.chunk_id],
            embeddings = [emb],
            documents  = [chunk.text],
            metadatas  = [flat],
        )

    print(f"[build_db] ✓ Collection now has {collection.count()} chunks.")
    return collection


if __name__ == "__main__":
    import argparse
    parser = argparse.ArgumentParser()
    parser.add_argument("--rebuild", action="store_true")
    args = parser.parse_args()
    build(force_rebuild=args.rebuild)
'''

with open("/kaggle/working/neva/rag/build_db.py", "w") as f:
    f.write(content)

import os
exists = os.path.exists("/kaggle/working/neva/rag/build_db.py")
print(f"✓ build_db.py written: {exists}")
print(f"✓ Collection already has 42 chunks — no rebuild needed")
print(f"✓ Safe to re-run Cell 11 now")

✓ build_db.py written: True
✓ Collection already has 42 chunks — no rebuild needed
✓ Safe to re-run Cell 11 now


In [13]:
# Cell 11 — Verify everything is cached for offline use
# This must pass before you disconnect from Kaggle internet

import sys, os
sys.path.insert(0, "/kaggle/working/neva")

print("="*65)
print("NEVA RAG — Offline Readiness Check")
print("="*65)

checks = []

# 1. ChromaDB collection exists and has chunks
try:
    import chromadb
    from chromadb.config import Settings
    client     = chromadb.PersistentClient(
        path="/kaggle/working/neva/chroma_db",
        settings=Settings(anonymized_telemetry=False),
    )
    collection = client.get_collection("neva_protocols")
    count      = collection.count()
    checks.append((f"ChromaDB collection ({count} chunks)", count > 0))
except Exception as e:
    checks.append((f"ChromaDB collection", False))
    print(f"  ERROR: {e}")

# 2. Embedding model is cached locally
try:
    from sentence_transformers import SentenceTransformer
    from pathlib import Path
    import os
    cache_dirs = [
        Path.home() / ".cache" / "huggingface" / "hub",
        Path.home() / ".cache" / "torch" / "sentence_transformers",
    ]
    model_cached = any(
        any("bge-m3" in str(p) for p in d.rglob("*") if d.exists())
        for d in cache_dirs
    )
    checks.append(("BAAI/bge-m3 model cached", model_cached))
except Exception as e:
    checks.append(("BAAI/bge-m3 model cached", False))

# 3. All required files exist
required_files = [
    "/kaggle/working/neva/rag/__init__.py",
    "/kaggle/working/neva/rag/models.py",
    "/kaggle/working/neva/rag/seed_protocols.py",
    "/kaggle/working/neva/rag/seed_protocols_ne.py",
    "/kaggle/working/neva/rag/build_db.py",
    "/kaggle/working/neva/rag/retriever.py",
    "/kaggle/working/neva/rag/rag_endpoint.py",
]
for fpath in required_files:
    exists = os.path.exists(fpath)
    checks.append((f"File: {os.path.basename(fpath)}", exists))

# 4. Retriever works without internet (use cached model)
try:
    from rag.retriever import retrieve
    from rag.models    import RetrievalRequest
    r = retrieve(RetrievalRequest(
        query    = "choking cannot breathe",
        language = "en",
        top_k    = 2,
    ))
    checks.append(("Retriever returns results", len(r.chunks) > 0))
    checks.append(("Retrieval is grounded",     r.grounded))
except Exception as e:
    checks.append(("Retriever works offline",   False))
    print(f"  ERROR: {e}")

# ── Print results ─────────────────────────────────────────────────────────────
print()
all_pass = True
for label, status in checks:
    icon = "✓" if status else "✗"
    print(f"  {icon}  {label}")
    if not status:
        all_pass = False

print()
if all_pass:
    print("✓ ALL CHECKS PASSED — RAG is fully offline-ready.")
    print()
    print("Collection summary:")
    meta = collection.get(include=["metadatas"])["metadatas"]
    from collections import Counter
    cond_counts = Counter(m["condition"] for m in meta)
    lang_counts = Counter(m["language"]  for m in meta)
    type_counts = Counter(m["step_type"] for m in meta)
    print(f"  Total chunks : {count}")
    print(f"  By language  : {dict(lang_counts)}")
    print(f"  By step_type : {dict(type_counts)}")
    print(f"  By condition :")
    for cond, n in sorted(cond_counts.items()):
        print(f"    {cond:30s} {n}")
else:
    print("✗ SOME CHECKS FAILED — fix before going offline.")

print()
print("="*65)
print("RAG WORKPACKAGE COMPLETE")
print("="*65)
print()
print("Integration contract for Gemma stage:")
print()
print("  from rag.models    import RetrievalRequest")
print("  from rag.retriever import retrieve, format_for_prompt")
print()
print("  result       = retrieve(RetrievalRequest(")
print("      query         = extracted_query,")
print("      condition     = extracted_condition,")
print("      urgency_level = extracted_urgency,")
print("      age_group     = extracted_age_group,")
print("      language      = detected_language,   # 'en' or 'ne'")
print("      top_k         = 5,")
print("  ))")
print()
print("  prompt_block = format_for_prompt(result, max_chunks=4)")
print("  # inject prompt_block into Gemma system prompt")
print()
print("  RAG API:  POST http://localhost:8001/rag/retrieve")
print("            POST http://localhost:8001/rag/prompt-block")

NEVA RAG — Offline Readiness Check

  ✓  ChromaDB collection (42 chunks)
  ✓  BAAI/bge-m3 model cached
  ✓  File: __init__.py
  ✓  File: models.py
  ✓  File: seed_protocols.py
  ✓  File: seed_protocols_ne.py
  ✓  File: build_db.py
  ✓  File: retriever.py
  ✓  File: rag_endpoint.py
  ✓  Retriever returns results
  ✓  Retrieval is grounded

✓ ALL CHECKS PASSED — RAG is fully offline-ready.

Collection summary:
  Total chunks : 42
  By language  : {'en': 22, 'ne': 20}
  By step_type : {'assessment': 8, 'action': 25, 'warning': 9}
  By condition :
    altitude_sickness              4
    burns                          4
    chest_pain                     4
    choking                        6
    drowning                       2
    seizure                        3
    severe_bleeding                4
    shock                          2
    snakebite                      5
    stroke                         4
    unconscious                    4

RAG WORKPACKAGE COMPLETE

Integration cont